# Phase 5 Follow-Up: Feature-Family Ablations

This notebook is a follow-up to `05_0_ml_feature_pipeline.ipynb`.

The goal here is to complete the missing explanatory part of Phase 5:

- run **feature-family ablations** on the best standalone ML model
- understand which feature groups are actually responsible for the standalone ML gains
- keep hybrid results from `05_0` as promising candidates, but treat them as provisional because their routing logic was designed after inspecting validation diagnostics

This notebook therefore focuses on **standalone model explanation**, not further hybrid-rule search.

## Main target model
Primary ablation target:
- `histgb_residual_full`

Reference models:
- `structured_winner`
- optionally `ridge_direct_full`

## Main questions
1. How much do structured-model prediction features matter?
2. How much do same-date anchor/value features matter?
3. How much do historical priors matter?
4. How much do date-level regime proxy features matter?

The output of this notebook should be a cleaner Phase 5 interpretation before any final Phase 6-style handoff.


In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)


## Notebook scope

This notebook does **not** introduce new model families or new hybrid routing rules.

It uses the locked Phase 5 setup from `05_0` and focuses only on:

- rebuilding the leakage-safe train / validation tables
- evaluating the standalone `histgb_residual_full` setup
- ablating feature families one at a time
- comparing protocol-level and fold-level impacts

The hybrid leader from `05_0` remains a strong candidate, but is not the object of explanation here.


In [2]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        has_root_files = (candidate / "train.csv").exists() and (candidate / "test.csv").exists()
        has_data_files = (candidate / "data" / "train.csv").exists() and (candidate / "data" / "test.csv").exists()
        if has_root_files or has_data_files:
            return candidate
    raise FileNotFoundError("Could not locate project root.")


PROJECT_ROOT = find_project_root()


def resolve_data_path(filename: str) -> Path:
    for path in [PROJECT_ROOT / filename, PROJECT_ROOT / "data" / filename]:
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {filename} in project root or data/")


train = pd.read_csv(resolve_data_path("train.csv"), parse_dates=["date"])
test = pd.read_csv(resolve_data_path("test.csv"), parse_dates=["date"])

for df in (train, test):
    if "tau" not in df.columns:
        df["tau"] = df["maturity_days"] / 365.0

train_date_summary = (
    train.groupby("date")
    .agg(
        total_rows=("row_id", "size"),
        observed_rows=("iv_observed", lambda s: s.notna().sum()),
        missing_rows=("iv_observed", lambda s: s.isna().sum()),
        observed_ratio=("iv_observed", lambda s: s.notna().mean()),
    )
    .sort_index()
)

train_dates = train_date_summary.index.to_list()

N_FOLDS = 4
VAL_DATES_PER_FOLD = 5
TOTAL_VAL_DATES = N_FOLDS * VAL_DATES_PER_FOLD
val_start_idx = len(train_dates) - TOTAL_VAL_DATES

fold_plan = pd.DataFrame(
    [
        {
            "fold": fold_id + 1,
            "train_start": train_dates[0],
            "train_end": train_dates[val_start_idx + fold_id * VAL_DATES_PER_FOLD - 1],
            "n_train_dates": val_start_idx + fold_id * VAL_DATES_PER_FOLD,
            "val_start": train_dates[val_start_idx + fold_id * VAL_DATES_PER_FOLD],
            "val_end": train_dates[val_start_idx + (fold_id + 1) * VAL_DATES_PER_FOLD - 1],
            "n_val_dates": VAL_DATES_PER_FOLD,
        }
        for fold_id in range(N_FOLDS)
    ]
)


In [4]:
BUCKET_COLS = ["maturity_label", "option_type"]
NODE_COLS = ["maturity_label", "moneyness", "option_type"]

MASK_PROTOCOL_CONFIG = {
    "stress_test": {
        "base_hide_rate": 0.10,
        "node_weight": 1.00,
        "support_weight": 0.00,
    },
    "primary_realistic": {
        "base_hide_rate": 0.10,
        "node_weight": 0.65,
        "support_weight": 0.35,
    },
}

LOCKED_MASK_SEED = "nqfo-val-v1"
PHASE4_SHRINK_ALPHA = 0.25

OVERALL_TEST_MISSING_RATE = test["iv_observed"].isna().mean()

test_bucket_pattern = (
    test.assign(is_missing=test["iv_observed"].isna())
    .groupby(BUCKET_COLS)["is_missing"]
    .mean()
    .rename("test_bucket_missing_rate")
    .reset_index()
)

test_node_pattern = (
    test.assign(is_missing=test["iv_observed"].isna())
    .groupby(NODE_COLS)["is_missing"]
    .mean()
    .rename("test_node_missing_rate")
    .reset_index()
)

surface_levels = pd.concat(
    [
        train[["moneyness", "maturity_label", "maturity_days"]],
        test[["moneyness", "maturity_label", "maturity_days"]],
    ],
    ignore_index=True,
)

moneyness_levels = sorted(surface_levels["moneyness"].dropna().unique().tolist())
maturity_levels = (
    surface_levels[["maturity_label", "maturity_days"]]
    .drop_duplicates()
    .sort_values("maturity_days")["maturity_label"]
    .tolist()
)

m_idx = {m: i for i, m in enumerate(moneyness_levels)}
t_idx = {t: i for i, t in enumerate(maturity_levels)}


def stable_uniform(key: str) -> float:
    digest = hashlib.md5(key.encode("utf-8")).hexdigest()
    return int(digest[:12], 16) / float(16**12 - 1)


def opposite_option(opt: str) -> str:
    return "put" if opt == "call" else "call"


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float))))


def local_support_profile(target_rows: pd.DataFrame, visible_rows: pd.DataFrame) -> pd.DataFrame:
    prof = target_rows.copy()

    visible_key_set = set(
        zip(
            visible_rows["date"],
            visible_rows["moneyness"],
            visible_rows["maturity_label"],
            visible_rows["option_type"],
        )
    )

    opp_visible = []
    same_maturity_adj_count = []
    same_moneyness_adj_count = []

    for d, m, t, o in zip(
        prof["date"],
        prof["moneyness"],
        prof["maturity_label"],
        prof["option_type"],
    ):
        opp_visible.append((d, m, t, opposite_option(o)) in visible_key_set)

        i = m_idx[m]
        j = t_idx[t]

        same_maturity_candidates = []
        if i - 1 >= 0:
            same_maturity_candidates.append((d, moneyness_levels[i - 1], t, o))
        if i + 1 < len(moneyness_levels):
            same_maturity_candidates.append((d, moneyness_levels[i + 1], t, o))

        same_moneyness_candidates = []
        if j - 1 >= 0:
            same_moneyness_candidates.append((d, m, maturity_levels[j - 1], o))
        if j + 1 < len(maturity_levels):
            same_moneyness_candidates.append((d, m, maturity_levels[j + 1], o))

        same_maturity_adj_count.append(sum(c in visible_key_set for c in same_maturity_candidates))
        same_moneyness_adj_count.append(sum(c in visible_key_set for c in same_moneyness_candidates))

    prof["opp_option_visible"] = opp_visible
    prof["same_maturity_adj_visible_count"] = same_maturity_adj_count
    prof["same_moneyness_adj_visible_count"] = same_moneyness_adj_count
    prof["local_support_score"] = (
        prof["opp_option_visible"].astype(int)
        + prof["same_maturity_adj_visible_count"]
        + prof["same_moneyness_adj_visible_count"]
    )
    return prof


def build_masked_validation_rows_with_protocol(
    full_df: pd.DataFrame,
    target_dates,
    protocol_name: str,
    seed: str = LOCKED_MASK_SEED,
) -> pd.DataFrame:
    cfg = MASK_PROTOCOL_CONFIG[protocol_name]

    val_df = full_df.loc[full_df["date"].isin(target_dates)].copy()
    val_df["is_orig_observed"] = val_df["iv_observed"].notna()
    val_df["is_orig_missing"] = ~val_df["is_orig_observed"]

    val_df = val_df.merge(test_bucket_pattern, on=BUCKET_COLS, how="left")
    val_df = val_df.merge(test_node_pattern, on=NODE_COLS, how="left")

    val_df["bucket_hide_rate_on_observed"] = (
        cfg["base_hide_rate"] * val_df["test_bucket_missing_rate"] / OVERALL_TEST_MISSING_RATE
    )

    val_df["priority_noise"] = val_df.apply(
        lambda r: stable_uniform(
            f"{seed}|{protocol_name}|{r['date'].date()}|{r['maturity_label']}|{r['moneyness']:.4f}|{r['option_type']}"
        ),
        axis=1,
    )

    observed_pool = val_df.loc[val_df["is_orig_observed"]].copy()
    observed_support = local_support_profile(observed_pool, observed_pool)[["row_id", "local_support_score"]]
    val_df = val_df.merge(observed_support, on="row_id", how="left")
    val_df["local_support_score"] = val_df["local_support_score"].fillna(0).astype(int)

    val_df["is_pseudo_hidden"] = False

    observed_pool = val_df.loc[val_df["is_orig_observed"]].copy()
    for _, g in observed_pool.groupby(["date", *BUCKET_COLS], sort=False):
        n_obs = len(g)
        n_hide = int(np.round(g["bucket_hide_rate_on_observed"].iloc[0] * n_obs))
        if n_hide <= 0:
            continue

        node_rank = g["test_node_missing_rate"].rank(method="average", pct=True)
        support_rank = g["local_support_score"].rank(method="average", pct=True)

        selection_priority = (
            cfg["node_weight"] * node_rank
            + cfg["support_weight"] * support_rank
            + 1e-6 * g["priority_noise"]
        )

        chosen_idx = (
            g.assign(selection_priority=selection_priority)
            .sort_values(["selection_priority", "row_id"], ascending=[False, True])
            .head(n_hide)
            .index
        )
        val_df.loc[chosen_idx, "is_pseudo_hidden"] = True

    val_df["is_scored_target"] = val_df["is_pseudo_hidden"]
    val_df["is_visible_anchor"] = val_df["is_orig_observed"] & ~val_df["is_pseudo_hidden"]
    val_df["is_effectively_missing"] = val_df["is_orig_missing"] | val_df["is_pseudo_hidden"]
    val_df["mask_protocol"] = protocol_name

    return val_df.sort_values(["date", "maturity_days", "option_type", "moneyness"]).reset_index(drop=True)


In [5]:
def build_node_lookup(observed_df: pd.DataFrame, pred_name: str) -> pd.DataFrame:
    return (
        observed_df.groupby(NODE_COLS)["iv_observed"]
        .median()
        .rename(pred_name)
        .reset_index()
    )


def predict_recent_node_median(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = target_rows.copy()

    observed_train = train_history.loc[train_history["iv_observed"].notna()].copy()
    global_median = observed_train["iv_observed"].median()

    recent_dates = sorted(observed_train["date"].unique())[-lookback_dates:]
    recent_obs = observed_train.loc[observed_train["date"].isin(recent_dates)].copy()

    recent_lookup = build_node_lookup(recent_obs, "recent_node_pred")
    full_lookup = build_node_lookup(observed_train, "full_node_pred")

    out = out.merge(recent_lookup, on=NODE_COLS, how="left")
    out = out.merge(full_lookup, on=NODE_COLS, how="left")

    out["iv_pred"] = out["recent_node_pred"].fillna(out["full_node_pred"]).fillna(global_median)
    out["pred_source"] = np.select(
        [
            out["recent_node_pred"].notna(),
            out["full_node_pred"].notna(),
        ],
        [
            "recent_node_median",
            "full_node_median",
        ],
        default="global_median",
    )
    return out


def predict_same_date_linear_interp(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = predict_recent_node_median(train_history, target_rows, lookback_dates=lookback_dates).copy()

    out["interp_pred"] = np.nan
    out["interp_source"] = pd.Series(index=out.index, dtype="object")

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["moneyness", "iv_observed"]]
            .dropna()
            .sort_values("moneyness")
        )

        if len(anchors) == 0:
            continue

        x = anchors["moneyness"].to_numpy()
        y = anchors["iv_observed"].to_numpy()

        if len(anchors) == 1:
            interp_vals = np.repeat(y[0], len(g))
            interp_label = "same_date_single_anchor"
        else:
            interp_vals = np.interp(g["moneyness"].to_numpy(), x, y)
            interp_label = "same_date_linear_interp"

        out.loc[g.index, "interp_pred"] = interp_vals
        out.loc[g.index, "interp_source"] = interp_label

    use_interp = out["interp_pred"].notna()
    out.loc[use_interp, "iv_pred"] = out.loc[use_interp, "interp_pred"]
    out["pred_source"] = np.where(use_interp, out["interp_source"], out["pred_source"])

    return out


def predict_quadratic_smile_logm(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
    degree: int = 2,
    min_anchors: int = 5,
) -> pd.DataFrame:
    out = predict_same_date_linear_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    out["log_moneyness"] = np.log(out["moneyness"])
    out["smile_pred"] = np.nan
    out["smile_anchor_count"] = 0
    out["pred_source_smile"] = pd.Series(index=out.index, dtype="object")

    observed_train = train_history.loc[train_history["iv_observed"].notna(), "iv_observed"]
    pred_lo = observed_train.quantile(0.001)
    pred_hi = observed_train.quantile(0.999)

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["log_moneyness", "iv_observed"]]
            .dropna()
            .sort_values("log_moneyness")
        )

        if len(anchors) < min_anchors:
            continue
        if anchors["log_moneyness"].nunique() < degree + 1:
            continue

        x = anchors["log_moneyness"].to_numpy()
        y = anchors["iv_observed"].to_numpy()

        x_center = x.mean()
        coeffs = np.polyfit(x - x_center, y, deg=degree)

        target_x = g["log_moneyness"].to_numpy()
        pred = np.polyval(coeffs, target_x - x_center)

        in_range = (target_x >= x.min()) & (target_x <= x.max())
        pred = np.where(in_range, pred, np.nan)
        pred = np.clip(pred, pred_lo, pred_hi)

        out.loc[g.index, "smile_pred"] = pred
        out.loc[g.index, "smile_anchor_count"] = len(anchors)
        out.loc[g.index, "pred_source_smile"] = np.where(in_range, "quadratic_smile_logm", pd.NA)

    use_smile = out["smile_pred"].notna()
    out.loc[use_smile, "iv_pred"] = out.loc[use_smile, "smile_pred"]
    out["pred_source"] = np.where(use_smile, out["pred_source_smile"], out["pred_source"])

    return out


def predict_total_variance_maturity_interp(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = predict_same_date_linear_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    out["anchor_total_variance"] = np.where(
        out["iv_observed"].notna(),
        (out["iv_observed"] / 100.0) ** 2 * out["tau"],
        np.nan,
    )
    out["tv_pred"] = np.nan
    out["tv_anchor_count"] = 0
    out["pred_source_tv"] = pd.Series(index=out.index, dtype="object")

    observed_train = train_history.loc[train_history["iv_observed"].notna(), "iv_observed"]
    pred_lo = observed_train.quantile(0.001)
    pred_hi = observed_train.quantile(0.999)

    for (_, moneyness, option_type), g_idx in out.groupby(
        ["date", "moneyness", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["tau", "anchor_total_variance"]]
            .dropna()
            .sort_values("tau")
        )

        if len(anchors) < 2:
            continue

        x = anchors["tau"].to_numpy()
        y = anchors["anchor_total_variance"].to_numpy()
        target_tau = g["tau"].to_numpy()

        in_range = (target_tau >= x.min()) & (target_tau <= x.max())
        pred_tv = np.interp(target_tau, x, y)
        pred_tv = np.where(in_range, pred_tv, np.nan)
        pred_tv = np.where(pred_tv > 0, pred_tv, np.nan)

        pred_iv = 100.0 * np.sqrt(pred_tv / target_tau)
        pred_iv = np.clip(pred_iv, pred_lo, pred_hi)

        out.loc[g.index, "tv_pred"] = pred_iv
        out.loc[g.index, "tv_anchor_count"] = len(anchors)
        out.loc[g.index, "pred_source_tv"] = np.where(in_range, "tv_maturity_interp", pd.NA)

    use_tv = out["tv_pred"].notna()
    out.loc[use_tv, "iv_pred"] = out.loc[use_tv, "tv_pred"]
    out["pred_source"] = np.where(use_tv, out["pred_source_tv"], out["pred_source"])

    return out


def predict_structured_region_blend(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
    smile_weight_center: float = 0.65,
    smile_weight_wing: float = 0.35,
) -> pd.DataFrame:
    base = predict_same_date_linear_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    smile = predict_quadratic_smile_logm(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    )[["row_id", "smile_pred"]].copy()

    tv = predict_total_variance_maturity_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    )[["row_id", "tv_pred"]].copy()

    out = base.merge(smile, on="row_id", how="left")
    out = out.merge(tv, on="row_id", how="left")

    out["wing_center_bucket"] = np.where(
        out["moneyness"].between(0.9, 1.1, inclusive="both"),
        "center",
        "wing",
    )

    both_available = out["smile_pred"].notna() & out["tv_pred"].notna()
    only_smile = out["smile_pred"].notna() & out["tv_pred"].isna()
    only_tv = out["tv_pred"].notna() & out["smile_pred"].isna()

    center_mask = both_available & (out["wing_center_bucket"] == "center")
    wing_mask = both_available & (out["wing_center_bucket"] == "wing")

    out.loc[center_mask, "iv_pred"] = (
        smile_weight_center * out.loc[center_mask, "smile_pred"]
        + (1.0 - smile_weight_center) * out.loc[center_mask, "tv_pred"]
    )
    out.loc[wing_mask, "iv_pred"] = (
        smile_weight_wing * out.loc[wing_mask, "smile_pred"]
        + (1.0 - smile_weight_wing) * out.loc[wing_mask, "tv_pred"]
    )
    out.loc[only_smile, "iv_pred"] = out.loc[only_smile, "smile_pred"]
    out.loc[only_tv, "iv_pred"] = out.loc[only_tv, "tv_pred"]

    out["pred_source"] = np.select(
        [
            center_mask,
            wing_mask,
            only_smile,
            only_tv,
        ],
        [
            "structured_region_blend_center",
            "structured_region_blend_wing",
            "quadratic_smile_only",
            "tv_maturity_only",
        ],
        default=out["pred_source"],
    )

    return out


def predict_structured_region_blend_callput_shrink(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
    shrink_alpha: float = PHASE4_SHRINK_ALPHA,
) -> pd.DataFrame:
    out = predict_structured_region_blend(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    opposite_visible = out.loc[
        out["is_visible_anchor"],
        ["date", "moneyness", "maturity_label", "option_type", "iv_observed"],
    ].copy()
    opposite_visible["option_type"] = opposite_visible["option_type"].map(opposite_option)
    opposite_visible = opposite_visible.rename(columns={"iv_observed": "opp_visible_iv"})

    out = out.merge(
        opposite_visible,
        on=["date", "moneyness", "maturity_label", "option_type"],
        how="left",
    )

    use_shrink = out["opp_visible_iv"].notna()
    out.loc[use_shrink, "iv_pred"] = (
        (1.0 - shrink_alpha) * out.loc[use_shrink, "iv_pred"]
        + shrink_alpha * out.loc[use_shrink, "opp_visible_iv"]
    )

    out["pred_source"] = np.where(
        use_shrink,
        "structured_region_blend_callput_shrink",
        out["pred_source"],
    )

    return out


In [6]:
ML_MIN_HISTORY_DATES = 20
ML_MASK_DATES_PER_FOLD = 20


def get_ml_candidate_dates_for_fold(
    fold_row,
    min_history_dates: int = ML_MIN_HISTORY_DATES,
    n_mask_dates: int = ML_MASK_DATES_PER_FOLD,
):
    fold_train_dates = train_date_summary.loc[:fold_row.train_end].index.to_list()
    eligible = fold_train_dates[min_history_dates:]
    return eligible[-n_mask_dates:]


def make_single_date_training_bundle(
    train_df: pd.DataFrame,
    target_date,
    protocol_name: str,
) -> dict:
    history_df = train_df.loc[train_df["date"] < target_date].copy()
    masked_target_date = build_masked_validation_rows_with_protocol(
        train_df,
        [target_date],
        protocol_name=protocol_name,
    )

    if history_df.empty:
        raise ValueError("History dataframe is empty. Choose a later target date.")

    if not (history_df["date"].max() < pd.Timestamp(target_date)):
        raise ValueError("Temporal leakage detected: history includes target date or later.")

    return {
        "target_date": pd.Timestamp(target_date),
        "protocol": protocol_name,
        "train_history": history_df,
        "masked_target_date": masked_target_date,
    }


def add_true_support_columns(scored_df: pd.DataFrame) -> pd.DataFrame:
    out = scored_df.copy()

    if "local_support_score" in out.columns:
        out = out.rename(columns={"local_support_score": "mask_local_support_score"})

    scored_targets = out.loc[out["is_scored_target"]].copy()
    visible_anchors = out.loc[out["is_visible_anchor"]].copy()

    support = local_support_profile(scored_targets, visible_anchors)[
        [
            "row_id",
            "opp_option_visible",
            "same_maturity_adj_visible_count",
            "same_moneyness_adj_visible_count",
            "local_support_score",
        ]
    ].copy()

    support = support.rename(columns={"local_support_score": "true_local_support_score"})
    support["any_local_same_date_support"] = (
        support["opp_option_visible"]
        | (support["same_maturity_adj_visible_count"] > 0)
        | (support["same_moneyness_adj_visible_count"] > 0)
    )
    support["hard_case"] = ~support["any_local_same_date_support"]

    out = out.merge(support, on="row_id", how="left")

    out["opp_option_visible"] = out["opp_option_visible"].astype("boolean").fillna(False).astype(bool)
    out["same_maturity_adj_visible_count"] = out["same_maturity_adj_visible_count"].fillna(0).astype(int)
    out["same_moneyness_adj_visible_count"] = out["same_moneyness_adj_visible_count"].fillna(0).astype(int)
    out["true_local_support_score"] = out["true_local_support_score"].fillna(0).astype(int)
    out["any_local_same_date_support"] = out["any_local_same_date_support"].astype("boolean").fillna(False).astype(bool)
    out["hard_case"] = out["hard_case"].astype("boolean").fillna(False).astype(bool)

    if "mask_local_support_score" in out.columns:
        out["mask_local_support_score"] = out["mask_local_support_score"].fillna(0).astype(int)

    return out


def add_same_maturity_anchor_features(masked_df: pd.DataFrame) -> pd.DataFrame:
    out = masked_df.copy()

    out["left_anchor_iv"] = np.nan
    out["right_anchor_iv"] = np.nan
    out["left_anchor_dist"] = np.nan
    out["right_anchor_dist"] = np.nan
    out["same_maturity_visible_anchor_count"] = 0

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["moneyness", "iv_observed"]]
            .dropna()
            .sort_values("moneyness")
        )

        if len(anchors) == 0:
            continue

        anchor_x = anchors["moneyness"].to_numpy()
        anchor_y = anchors["iv_observed"].to_numpy()
        target_x = g["moneyness"].to_numpy()

        left_iv = []
        right_iv = []
        left_dist = []
        right_dist = []

        for x in target_x:
            left_mask = anchor_x <= x
            right_mask = anchor_x >= x

            if left_mask.any():
                lx = anchor_x[left_mask][-1]
                ly = anchor_y[left_mask][-1]
                left_iv.append(ly)
                left_dist.append(abs(x - lx))
            else:
                left_iv.append(np.nan)
                left_dist.append(np.nan)

            if right_mask.any():
                rx = anchor_x[right_mask][0]
                ry = anchor_y[right_mask][0]
                right_iv.append(ry)
                right_dist.append(abs(rx - x))
            else:
                right_iv.append(np.nan)
                right_dist.append(np.nan)

        out.loc[g.index, "left_anchor_iv"] = left_iv
        out.loc[g.index, "right_anchor_iv"] = right_iv
        out.loc[g.index, "left_anchor_dist"] = left_dist
        out.loc[g.index, "right_anchor_dist"] = right_dist
        out.loc[g.index, "same_maturity_visible_anchor_count"] = len(anchors)

    return out


def add_same_node_opposite_option_feature(masked_df: pd.DataFrame) -> pd.DataFrame:
    out = masked_df.copy()

    opp_visible = (
        out.loc[out["is_visible_anchor"], ["date", "moneyness", "maturity_label", "option_type", "iv_observed"]]
        .copy()
    )
    opp_visible["option_type"] = opp_visible["option_type"].map(opposite_option)
    opp_visible = opp_visible.rename(columns={"iv_observed": "opp_visible_iv_same_node"})

    out = out.merge(
        opp_visible,
        on=["date", "moneyness", "maturity_label", "option_type"],
        how="left",
    )

    return out


def get_nearest_moneyness_rows(df: pd.DataFrame, target: float) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    distances = (df["moneyness"] - target).abs()
    min_dist = distances.groupby(df["date"]).transform("min")
    return df.loc[min_dist.eq(distances)].copy()


def add_date_level_regime_proxy_features(masked_df: pd.DataFrame) -> pd.DataFrame:
    out = masked_df.copy()
    visible = out.loc[out["is_visible_anchor"] & out["iv_observed"].notna()].copy()

    total_rows = out.groupby("date").size().rename("date_total_row_count")
    visible_count = visible.groupby("date").size().rename("date_visible_anchor_count")
    visible_ratio = (visible_count / total_rows).rename("date_visible_anchor_ratio")
    visible_mean = visible.groupby("date")["iv_observed"].mean().rename("date_visible_iv_mean")
    visible_dispersion = visible.groupby("date")["iv_observed"].std().rename("date_visible_iv_dispersion")

    atm = (
        get_nearest_moneyness_rows(visible, 1.0)
        .groupby("date")["iv_observed"]
        .mean()
        .rename("date_atm_iv_proxy")
    )
    left = (
        get_nearest_moneyness_rows(visible, 0.9)
        .groupby("date")["iv_observed"]
        .mean()
        .rename("date_iv_0p9_proxy")
    )
    right = (
        get_nearest_moneyness_rows(visible, 1.1)
        .groupby("date")["iv_observed"]
        .mean()
        .rename("date_iv_1p1_proxy")
    )

    maturity_means = (
        visible.groupby(["date", "maturity_days"])["iv_observed"]
        .mean()
        .reset_index()
        .sort_values(["date", "maturity_days"])
    )
    short_end = (
        maturity_means.groupby("date").first()["iv_observed"]
        .rename("date_short_end_iv_proxy")
    )
    long_end = (
        maturity_means.groupby("date").last()["iv_observed"]
        .rename("date_long_end_iv_proxy")
    )

    proxy_df = pd.concat(
        [
            total_rows,
            visible_count,
            visible_ratio,
            visible_mean,
            visible_dispersion,
            atm,
            left,
            right,
            short_end,
            long_end,
        ],
        axis=1,
    ).reset_index()

    proxy_df["date_visible_anchor_count"] = proxy_df["date_visible_anchor_count"].fillna(0).astype(int)
    proxy_df["date_visible_anchor_ratio"] = proxy_df["date_visible_anchor_ratio"].fillna(0.0)
    proxy_df["date_skew_proxy"] = proxy_df["date_iv_0p9_proxy"] - proxy_df["date_iv_1p1_proxy"]
    proxy_df["date_term_slope_proxy"] = (
        proxy_df["date_short_end_iv_proxy"] - proxy_df["date_long_end_iv_proxy"]
    )

    out = out.merge(proxy_df, on="date", how="left")
    return out


def build_feature_table_for_masked_bundle(
    train_history: pd.DataFrame,
    masked_target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    base = masked_target_rows.copy()

    linear_pred = predict_same_date_linear_interp(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "iv_pred"]
    ].rename(columns={"iv_pred": "same_date_linear_interp_pred"})

    smile_pred = predict_quadratic_smile_logm(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "smile_pred"]
    ].rename(columns={"smile_pred": "quadratic_smile_logm_pred"})

    tv_pred = predict_total_variance_maturity_interp(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "tv_pred"]
    ].rename(columns={"tv_pred": "tv_maturity_interp_pred"})

    region_pred = predict_structured_region_blend(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "iv_pred", "pred_source"]
    ].rename(
        columns={
            "iv_pred": "structured_region_blend_pred",
            "pred_source": "structured_region_blend_source",
        }
    )

    winner_pred = predict_structured_region_blend_callput_shrink(
        train_history,
        base,
        lookback_dates=lookback_dates,
        shrink_alpha=PHASE4_SHRINK_ALPHA,
    )[["row_id", "iv_pred", "pred_source"]].rename(
        columns={
            "iv_pred": "structured_winner_pred",
            "pred_source": "structured_winner_source",
        }
    )

    node_pred = predict_recent_node_median(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "recent_node_pred", "full_node_pred"]
    ]

    feat = base.merge(linear_pred, on="row_id", how="left")
    feat = feat.merge(smile_pred, on="row_id", how="left")
    feat = feat.merge(tv_pred, on="row_id", how="left")
    feat = feat.merge(region_pred, on="row_id", how="left")
    feat = feat.merge(winner_pred, on="row_id", how="left")
    feat = feat.merge(node_pred, on="row_id", how="left")

    feat = add_true_support_columns(feat)
    feat = add_same_maturity_anchor_features(feat)
    feat = add_same_node_opposite_option_feature(feat)
    feat = add_date_level_regime_proxy_features(feat)

    feat["support_score_gap"] = feat["true_local_support_score"] - feat.get("mask_local_support_score", 0)
    feat["has_opp_same_node_visible"] = feat["opp_visible_iv_same_node"].notna().astype(int)

    feat["log_moneyness"] = np.log(feat["moneyness"])
    feat["is_call"] = (feat["option_type"] == "call").astype(int)
    feat["wing_center_bucket"] = np.where(
        feat["moneyness"].between(0.9, 1.1, inclusive="both"),
        "center",
        "wing",
    )
    feat["is_center"] = (feat["wing_center_bucket"] == "center").astype(int)
    feat["is_wing"] = (feat["wing_center_bucket"] == "wing").astype(int)
    feat["is_edge_maturity"] = feat["maturity_label"].isin(["1M", "6M"]).astype(int)
    feat["is_interior_maturity"] = 1 - feat["is_edge_maturity"]

    feat["smile_minus_linear"] = feat["quadratic_smile_logm_pred"] - feat["same_date_linear_interp_pred"]
    feat["tv_minus_linear"] = feat["tv_maturity_interp_pred"] - feat["same_date_linear_interp_pred"]
    feat["winner_minus_linear"] = feat["structured_winner_pred"] - feat["same_date_linear_interp_pred"]
    feat["winner_minus_region"] = feat["structured_winner_pred"] - feat["structured_region_blend_pred"]

    feat["target_iv"] = feat["iv_observed"]
    feat["target_residual"] = feat["target_iv"] - feat["structured_winner_pred"]

    return feat


In [7]:
def build_training_table_for_fold_protocol(
    train_df: pd.DataFrame,
    fold_row,
    protocol_name: str,
    lookback_dates: int = 20,
    min_history_dates: int = ML_MIN_HISTORY_DATES,
    n_mask_dates: int = ML_MASK_DATES_PER_FOLD,
) -> pd.DataFrame:
    target_dates = get_ml_candidate_dates_for_fold(
        fold_row,
        min_history_dates=min_history_dates,
        n_mask_dates=n_mask_dates,
    )

    tables = []
    for target_date in target_dates:
        bundle = make_single_date_training_bundle(
            train_df,
            target_date=target_date,
            protocol_name=protocol_name,
        )

        feat = build_feature_table_for_masked_bundle(
            bundle["train_history"],
            bundle["masked_target_date"],
            lookback_dates=lookback_dates,
        )

        scored = feat.loc[feat["is_scored_target"]].copy()
        if not scored["is_orig_observed"].all():
            raise ValueError("Pseudo-masked training examples must come from originally observed rows only.")

        scored["target_date"] = pd.Timestamp(target_date)
        scored["protocol"] = protocol_name
        scored["fold"] = fold_row.fold
        scored["n_history_dates"] = bundle["train_history"]["date"].nunique()
        tables.append(scored)

    return pd.concat(tables, ignore_index=True)


def build_validation_table_for_fold_protocol(
    train_df: pd.DataFrame,
    fold_row,
    protocol_name: str,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    val_dates = train_date_summary.loc[fold_row.val_start:fold_row.val_end].index.to_list()
    train_history = train_df.loc[train_df["date"] <= fold_row.train_end].copy()
    masked_val_rows = build_masked_validation_rows_with_protocol(
        train_df,
        val_dates,
        protocol_name=protocol_name,
    )

    feat = build_feature_table_for_masked_bundle(
        train_history,
        masked_val_rows,
        lookback_dates=lookback_dates,
    )

    scored = feat.loc[feat["is_scored_target"]].copy()
    if not scored["is_orig_observed"].all():
        raise ValueError("Validation scored rows must come from originally observed rows only.")

    scored["target_date"] = scored["date"]
    scored["protocol"] = protocol_name
    scored["fold"] = fold_row.fold
    scored["n_history_dates"] = train_history["date"].nunique()
    return scored.reset_index(drop=True)


def summarize_feature_table(feature_table: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "n_rows": len(feature_table),
            "n_dates": feature_table["target_date"].nunique(),
            "date_min": feature_table["target_date"].min().date().isoformat(),
            "date_max": feature_table["target_date"].max().date().isoformat(),
            "target_iv_missing": int(feature_table["target_iv"].isna().sum()),
            "target_residual_missing": int(feature_table["target_residual"].isna().sum()),
            "winner_pred_missing": int(feature_table["structured_winner_pred"].isna().sum()),
            "mean_target_iv": feature_table["target_iv"].mean(),
            "mean_target_residual": feature_table["target_residual"].mean(),
            "rmse_structured_winner": rmse(feature_table["target_iv"], feature_table["structured_winner_pred"]),
        }
    )


def schema_alignment_report(train_table: pd.DataFrame, val_table: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:
    train_cols = set(train_table.columns)
    val_cols = set(val_table.columns)
    train_only = sorted(train_cols - val_cols)
    val_only = sorted(val_cols - train_cols)

    summary = pd.Series(
        {
            "n_train_cols": len(train_cols),
            "n_val_cols": len(val_cols),
            "n_common_cols": len(train_cols & val_cols),
            "n_train_only_cols": len(train_only),
            "n_val_only_cols": len(val_only),
            "schemas_match": len(train_only) == 0 and len(val_only) == 0,
        }
    )
    detail = pd.DataFrame(
        {
            "train_only_cols": pd.Series(train_only, dtype="object"),
            "val_only_cols": pd.Series(val_only, dtype="object"),
        }
    )
    return summary, detail


In [8]:
def rmse_from_series(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def prepare_design_matrices(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
):
    X_train = train_df.loc[:, feature_cols].copy()
    X_val = val_df.loc[:, feature_cols].copy()

    for df in (X_train, X_val):
        bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
        for col in bool_cols:
            df[col] = df[col].astype(int)

    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X_train.columns if c not in cat_cols]

    if num_cols:
        train_num = X_train[num_cols].copy()
        val_num = X_val[num_cols].copy()

        medians = train_num.median(numeric_only=True)
        train_num = train_num.fillna(medians)
        val_num = val_num.fillna(medians)
    else:
        train_num = pd.DataFrame(index=X_train.index)
        val_num = pd.DataFrame(index=X_val.index)

    if cat_cols:
        train_cat = pd.get_dummies(X_train[cat_cols], dummy_na=True)
        val_cat = pd.get_dummies(X_val[cat_cols], dummy_na=True)
        train_cat, val_cat = train_cat.align(val_cat, join="outer", axis=1, fill_value=0)
    else:
        train_cat = pd.DataFrame(index=X_train.index)
        val_cat = pd.DataFrame(index=X_val.index)

    X_train_final = pd.concat([train_num, train_cat], axis=1)
    X_val_final = pd.concat([val_num, val_cat], axis=1)

    return X_train_final, X_val_final


def evaluate_ridge_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
    target_col: str,
    alpha: float = 1.0,
    prediction_mode: str = "direct_iv",
):
    X_train, X_val = prepare_design_matrices(train_df, val_df, feature_cols)
    y_train = train_df[target_col].copy()
    y_val = val_df["target_iv"].copy()

    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)

    raw_pred = model.predict(X_val)

    if prediction_mode == "residual":
        final_pred = val_df["structured_winner_pred"].to_numpy() + raw_pred
    elif prediction_mode == "direct_iv":
        final_pred = raw_pred
    else:
        raise ValueError(f"Unknown prediction_mode: {prediction_mode}")

    out = {
        "alpha": alpha,
        "target_col": target_col,
        "prediction_mode": prediction_mode,
        "n_features_after_encoding": X_train.shape[1],
        "train_target_mean": float(y_train.mean()),
        "val_rmse": rmse_from_series(y_val, final_pred),
        "structured_winner_rmse": rmse_from_series(val_df["target_iv"], val_df["structured_winner_pred"]),
    }

    pred_df = val_df[["row_id", "target_date", "target_iv", "structured_winner_pred"]].copy()
    pred_df["ml_raw_pred"] = raw_pred
    pred_df["iv_pred"] = final_pred

    return out, pred_df, model, X_train.columns.tolist()


def evaluate_histgb_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
    target_col: str,
    learning_rate: float = 0.05,
    max_depth: int = 3,
    max_iter: int = 300,
    min_samples_leaf: int = 10,
    prediction_mode: str = "direct_iv",
):
    X_train, X_val = prepare_design_matrices(train_df, val_df, feature_cols)
    y_train = train_df[target_col].copy()
    y_val = val_df["target_iv"].copy()

    model = HistGradientBoostingRegressor(
        learning_rate=learning_rate,
        max_depth=max_depth,
        max_iter=max_iter,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
    )
    model.fit(X_train, y_train)

    raw_pred = model.predict(X_val)

    if prediction_mode == "residual":
        final_pred = val_df["structured_winner_pred"].to_numpy() + raw_pred
    elif prediction_mode == "direct_iv":
        final_pred = raw_pred
    else:
        raise ValueError(f"Unknown prediction_mode: {prediction_mode}")

    out = {
        "target_col": target_col,
        "prediction_mode": prediction_mode,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "max_iter": max_iter,
        "min_samples_leaf": min_samples_leaf,
        "n_features_after_encoding": X_train.shape[1],
        "train_target_mean": float(y_train.mean()),
        "val_rmse": rmse_from_series(y_val, final_pred),
        "structured_winner_rmse": rmse_from_series(val_df["target_iv"], val_df["structured_winner_pred"]),
    }

    pred_df = val_df[
        [
            "row_id",
            "target_date",
            "option_type",
            "maturity_label",
            "maturity_days",
            "moneyness",
            "target_iv",
            "structured_winner_pred",
            "target_residual",
            "structured_winner_source",
        ]
    ].copy()
    pred_df["ml_raw_pred"] = raw_pred
    pred_df["iv_pred"] = final_pred

    return out, pred_df, model


In [9]:
print("Bootstrap check")
print("train shape:", train.shape)
print("test shape:", test.shape)
display(fold_plan)


Bootstrap check
train shape: (11640, 10)
test shape: (3960, 10)


,fold,train_start,train_end,n_train_dates,val_start,val_end,n_val_dates
0,1,2025-01-02,2025-04-18,77,2025-04-21,2025-04-25,5
1,2,2025-01-02,2025-04-25,82,2025-04-28,2025-05-02,5
2,3,2025-01-02,2025-05-02,87,2025-05-05,2025-05-09,5
3,4,2025-01-02,2025-05-09,92,2025-05-12,2025-05-16,5


In [10]:
FEATURE_GROUPS = {
    "row_local_core": [
        "moneyness",
        "log_moneyness",
        "maturity_days",
        "tau",
        "is_call",
        "is_center",
        "is_wing",
        "is_edge_maturity",
        "is_interior_maturity",
        "option_type",
        "maturity_label",
        "wing_center_bucket",
    ],
    "structured_predictions": [
        "structured_winner_pred",
        "structured_region_blend_pred",
        "tv_maturity_interp_pred",
        "quadratic_smile_logm_pred",
        "same_date_linear_interp_pred",
        "structured_winner_source",
        "structured_region_blend_source",
    ],
    "structured_gaps": [
        "smile_minus_linear",
        "tv_minus_linear",
        "winner_minus_linear",
        "winner_minus_region",
    ],
    "same_date_anchor_values": [
        "opp_visible_iv_same_node",
        "has_opp_same_node_visible",
        "left_anchor_iv",
        "right_anchor_iv",
        "left_anchor_dist",
        "right_anchor_dist",
        "same_maturity_visible_anchor_count",
    ],
    "support_geometry": [
        "opp_option_visible",
        "same_maturity_adj_visible_count",
        "same_moneyness_adj_visible_count",
        "true_local_support_score",
        "mask_local_support_score",
        "support_score_gap",
        "hard_case",
    ],
    "historical_priors": [
        "recent_node_pred",
        "full_node_pred",
    ],
    "date_regime_proxies": [
        "date_atm_iv_proxy",
        "date_skew_proxy",
        "date_term_slope_proxy",
        "date_visible_anchor_ratio",
        "date_visible_iv_dispersion",
        "date_visible_iv_mean",
        "date_visible_anchor_count",
    ],
}

FULL_HISTGB_ABLATION_FEATURES = sorted(
    {
        col
        for cols in FEATURE_GROUPS.values()
        for col in cols
    }
)

print("Number of grouped features:", len(FULL_HISTGB_ABLATION_FEATURES))
for group_name, cols in FEATURE_GROUPS.items():
    print(f"{group_name}: {len(cols)}")


Number of grouped features: 46
row_local_core: 12
structured_predictions: 7
structured_gaps: 4
same_date_anchor_values: 7
support_geometry: 7
historical_priors: 2
date_regime_proxies: 7


In [13]:
fold1_primary_train_table = build_training_table_for_fold_protocol(
    train,
    fold_plan.iloc[0],
    protocol_name="primary_realistic",
    lookback_dates=20,
)

fold1_primary_val_table = build_validation_table_for_fold_protocol(
    train,
    fold_plan.iloc[0],
    protocol_name="primary_realistic",
    lookback_dates=20,
)

print("Fold 1 / primary_realistic training table summary")
display(summarize_feature_table(fold1_primary_train_table).to_frame("value"))

print("Fold 1 / primary_realistic validation table summary")
display(summarize_feature_table(fold1_primary_val_table).to_frame("value"))

schema_summary, schema_detail = schema_alignment_report(
    fold1_primary_train_table,
    fold1_primary_val_table,
)

print("Train / validation schema alignment")
display(schema_summary.to_frame("value"))
display(schema_detail)


Fold 1 / primary_realistic training table summary


,value
n_rows,155
n_dates,20
date_min,2025-03-24
date_max,2025-04-18
target_iv_missing,0
target_residual_missing,0
winner_pred_missing,0
mean_target_iv,21.3784
mean_target_residual,0.0926
rmse_structured_winner,0.9005


Fold 1 / primary_realistic validation table summary


,value
n_rows,39
n_dates,5
date_min,2025-04-21
date_max,2025-04-25
target_iv_missing,0
target_residual_missing,0
winner_pred_missing,0
mean_target_iv,19.5990
mean_target_residual,0.3840
rmse_structured_winner,1.3592


Train / validation schema alignment


,value
n_train_cols,74
n_val_cols,74
n_common_cols,74
n_train_only_cols,0
n_val_only_cols,0
schemas_match,True


,train_only_cols,val_only_cols


In [14]:
missing_feature_cols = [
    col for col in FULL_HISTGB_ABLATION_FEATURES
    if col not in fold1_primary_train_table.columns
]

print("Missing grouped feature columns:", missing_feature_cols)


Missing grouped feature columns: []


In [12]:
PRIMARY_ABLATION_MODEL = "histgb_residual_full"
PRIMARY_ABLATION_TARGET = "target_residual"

print("Primary ablation model:", PRIMARY_ABLATION_MODEL)
print("Primary ablation target:", PRIMARY_ABLATION_TARGET)


Primary ablation model: histgb_residual_full
Primary ablation target: target_residual


In [15]:
histgb_fold1_primary_summary, histgb_fold1_primary_preds, histgb_fold1_primary_model = evaluate_histgb_model(
    fold1_primary_train_table,
    fold1_primary_val_table,
    feature_cols=FULL_HISTGB_ABLATION_FEATURES,
    target_col="target_residual",
    learning_rate=0.05,
    max_depth=3,
    max_iter=300,
    min_samples_leaf=10,
    prediction_mode="residual",
)

histgb_fold1_primary_result = pd.DataFrame(
    [
        {
            "model": "histgb_residual_full",
            **histgb_fold1_primary_summary,
        }
    ]
)

print("Fold 1 / primary_realistic standalone HistGB residual check")
display(histgb_fold1_primary_result)


Fold 1 / primary_realistic standalone HistGB residual check


,model,target_col,prediction_mode,learning_rate,max_depth,max_iter,min_samples_leaf,n_features_after_encoding,train_target_mean,val_rmse,structured_winner_rmse
0,histgb_residual_full,target_residual,residual,0.0500,3,300,10,65,0.0926,0.8903,1.3592


In [16]:
fold1_primary_compare = pd.DataFrame(
    [
        {
            "model": "structured_winner",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                fold1_primary_val_table["structured_winner_pred"],
            ),
        },
        {
            "model": "histgb_residual_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                histgb_fold1_primary_preds["iv_pred"],
            ),
        },
    ]
).sort_values("val_rmse")

print("Fold 1 / primary_realistic comparison")
display(fold1_primary_compare)


Fold 1 / primary_realistic comparison


,model,val_rmse
1,histgb_residual_full,0.8903
0,structured_winner,1.3592


In [17]:
fold1_primary_preview = fold1_primary_val_table[
    [
        "row_id",
        "target_date",
        "option_type",
        "maturity_label",
        "maturity_days",
        "moneyness",
        "target_iv",
        "structured_winner_pred",
        "target_residual",
        "structured_winner_source",
    ]
].copy()

fold1_primary_preview = fold1_primary_preview.merge(
    histgb_fold1_primary_preds[["row_id", "iv_pred"]].rename(
        columns={"iv_pred": "histgb_residual_iv_pred"}
    ),
    on="row_id",
    how="left",
)

print("Fold 1 / primary_realistic prediction preview")
display(
    fold1_primary_preview
    .sort_values(["target_date", "option_type", "maturity_days", "moneyness"])
    .reset_index(drop=True)
)


Fold 1 / primary_realistic prediction preview


,row_id,target_date,option_type,maturity_label,maturity_days,moneyness,target_iv,structured_winner_pred,target_residual,structured_winner_source,histgb_residual_iv_pred
0,9262,2025-04-21,call,1M,30,1.1000,19.2304,19.1951,0.0353,structured_region_blend_callput_shrink,19.2284
1,9294,2025-04-21,call,2M,60,1.1250,19.0918,18.4505,0.6413,structured_region_blend_callput_shrink,18.2645
2,9312,2025-04-21,call,3M,91,0.9750,18.7062,19.3098,-0.6036,structured_region_blend_callput_shrink,19.3050
3,9344,2025-04-21,call,6M,182,1.0000,17.1591,18.0928,-0.9337,structured_region_blend_callput_shrink,17.5824
4,9245,2025-04-21,put,1M,30,0.8750,24.2840,25.1451,-0.8611,quadratic_smile_only,25.7073
5,9281,2025-04-21,put,2M,60,0.9500,20.2648,20.5633,-0.2985,structured_region_blend_callput_shrink,21.0538
6,9329,2025-04-21,put,3M,91,1.2000,16.8226,17.9854,-1.1628,tv_maturity_only,17.3933
7,9333,2025-04-21,put,6M,182,0.8500,20.5319,19.8634,0.6685,structured_region_blend_callput_shrink,20.5784
8,9372,2025-04-22,call,1M,30,0.9750,21.0011,20.9739,0.0272,quadratic_smile_only,20.9259
9,9390,2025-04-22,call,2M,60,0.8000,24.7991,21.2919,3.5072,same_date_linear_interp,22.8439


In [18]:
PROTOCOLS = ["primary_realistic", "stress_test"]

table_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        train_table = build_training_table_for_fold_protocol(
            train,
            fold_row,
            protocol_name=protocol_name,
            lookback_dates=20,
        )

        val_table = build_validation_table_for_fold_protocol(
            train,
            fold_row,
            protocol_name=protocol_name,
            lookback_dates=20,
        )

        table_cache[(protocol_name, fold_id)] = {
            "train": train_table,
            "val": val_table,
        }

print("Cached tables")
display(
    pd.DataFrame(
        [
            {
                "protocol": protocol_name,
                "fold": fold_id,
                "n_train_rows": len(payload["train"]),
                "n_train_dates": payload["train"]["target_date"].nunique(),
                "n_val_rows": len(payload["val"]),
                "n_val_dates": payload["val"]["target_date"].nunique(),
                "schemas_match": set(payload["train"].columns) == set(payload["val"].columns),
            }
            for (protocol_name, fold_id), payload in table_cache.items()
        ]
    ).sort_values(["protocol", "fold"]).reset_index(drop=True)
)


Cached tables


,protocol,fold,n_train_rows,n_train_dates,n_val_rows,n_val_dates,schemas_match
0,primary_realistic,1,155,20,39,5,True
1,primary_realistic,2,155,20,38,5,True
2,primary_realistic,3,155,20,39,5,True
3,primary_realistic,4,156,20,40,5,True
4,stress_test,1,155,20,39,5,True
5,stress_test,2,155,20,38,5,True
6,stress_test,3,155,20,39,5,True
7,stress_test,4,156,20,40,5,True


In [19]:
def run_structured_winner_baseline(train_table: pd.DataFrame, val_table: pd.DataFrame):
    pred_df = val_table[
        [
            "row_id",
            "target_date",
            "target_iv",
            "structured_winner_pred",
            "structured_winner_source",
        ]
    ].copy()
    pred_df["iv_pred"] = pred_df["structured_winner_pred"]

    summary = {
        "model": "structured_winner",
        "target_col": "target_iv",
        "prediction_mode": "direct_iv",
        "n_features_after_encoding": 0,
        "val_rmse": rmse_from_series(val_table["target_iv"], pred_df["iv_pred"]),
    }
    return summary, pred_df


def run_ridge_direct_full(train_table: pd.DataFrame, val_table: pd.DataFrame):
    summary, pred_df, model, cols = evaluate_ridge_model(
        train_table,
        val_table,
        feature_cols=FULL_HISTGB_ABLATION_FEATURES,
        target_col="target_iv",
        alpha=1.0,
        prediction_mode="direct_iv",
    )
    summary = {"model": "ridge_direct_full", **summary}
    return summary, pred_df


def run_histgb_residual_full(train_table: pd.DataFrame, val_table: pd.DataFrame):
    summary, pred_df, model = evaluate_histgb_model(
        train_table,
        val_table,
        feature_cols=FULL_HISTGB_ABLATION_FEATURES,
        target_col="target_residual",
        learning_rate=0.05,
        max_depth=3,
        max_iter=300,
        min_samples_leaf=10,
        prediction_mode="residual",
    )
    summary = {"model": "histgb_residual_full", **summary}
    return summary, pred_df


BASELINE_RUNNERS = {
    "structured_winner": run_structured_winner_baseline,
    "ridge_direct_full": run_ridge_direct_full,
    "histgb_residual_full": run_histgb_residual_full,
}


In [20]:
baseline_rows = []
baseline_prediction_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        train_table = table_cache[(protocol_name, fold_id)]["train"]
        val_table = table_cache[(protocol_name, fold_id)]["val"]

        for model_name, runner in BASELINE_RUNNERS.items():
            summary, pred_df = runner(train_table, val_table)

            baseline_rows.append(
                {
                    "protocol": protocol_name,
                    "fold": fold_id,
                    "model": model_name,
                    "n_train_rows": len(train_table),
                    "n_val_rows": len(val_table),
                    "structured_winner_rmse": rmse_from_series(
                        val_table["target_iv"],
                        val_table["structured_winner_pred"],
                    ),
                    **summary,
                }
            )

            pred_df = pred_df.copy()
            pred_df["protocol"] = protocol_name
            pred_df["fold"] = fold_id
            pred_df["model"] = model_name
            baseline_prediction_cache[(protocol_name, fold_id, model_name)] = pred_df

baseline_results = (
    pd.DataFrame(baseline_rows)
    .sort_values(["protocol", "fold", "val_rmse", "model"])
    .reset_index(drop=True)
)

print("Standalone baseline results")
display(baseline_results)


Standalone baseline results


,protocol,fold,model,n_train_rows,n_val_rows,structured_winner_rmse,target_col,prediction_mode,n_features_after_encoding,val_rmse,alpha,train_target_mean,learning_rate,max_depth,max_iter,min_samples_leaf
0,primary_realistic,1,histgb_residual_full,155,39,1.3592,target_residual,residual,65,0.8903,NaN,0.0926,0.0500,3.0000,300.0000,10.0000
1,primary_realistic,1,ridge_direct_full,155,39,1.3592,target_iv,direct_iv,65,1.0556,1.0000,21.3784,NaN,NaN,NaN,NaN
2,primary_realistic,1,structured_winner,155,39,1.3592,target_iv,direct_iv,0,1.3592,NaN,NaN,NaN,NaN,NaN,NaN
3,primary_realistic,2,histgb_residual_full,155,38,1.0373,target_residual,residual,65,0.8908,NaN,0.1957,0.0500,3.0000,300.0000,10.0000
4,primary_realistic,2,ridge_direct_full,155,38,1.0373,target_iv,direct_iv,65,0.9294,1.0000,20.7935,NaN,NaN,NaN,NaN
5,primary_realistic,2,structured_winner,155,38,1.0373,target_iv,direct_iv,0,1.0373,NaN,NaN,NaN,NaN,NaN,NaN
6,primary_realistic,3,structured_winner,155,39,0.7840,target_iv,direct_iv,0,0.7840,NaN,NaN,NaN,NaN,NaN,NaN
7,primary_realistic,3,ridge_direct_full,155,39,0.7840,target_iv,direct_iv,65,0.8733,1.0000,21.9630,NaN,NaN,NaN,NaN
8,primary_realistic,3,histgb_residual_full,155,39,0.7840,target_residual,residual,65,0.8861,NaN,0.1308,0.0500,3.0000,300.0000,10.0000
9,primary_realistic,4,histgb_residual_full,156,40,1.0748,target_residual,residual,65,0.8171,NaN,0.2026,0.0500,3.0000,300.0000,10.0000


In [21]:
baseline_protocol_summary = (
    baseline_results
    .groupby(["protocol", "model"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_rmse"])
)

print("Standalone baseline protocol summary")
display(baseline_protocol_summary)


Standalone baseline protocol summary


,protocol,model,mean_rmse,min_rmse,max_rmse
0,primary_realistic,histgb_residual_full,0.8711,0.8171,0.8908
1,primary_realistic,ridge_direct_full,0.9245,0.8397,1.0556
2,primary_realistic,structured_winner,1.0638,0.7840,1.3592
4,stress_test,ridge_direct_full,0.8463,0.7246,1.0449
3,stress_test,histgb_residual_full,0.9424,0.8574,1.1223
5,stress_test,structured_winner,1.1847,1.0614,1.3163


In [22]:
baseline_protocol_pivot = (
    baseline_protocol_summary
    .pivot(index="model", columns="protocol", values="mean_rmse")
    .reset_index()
)

print("Standalone baseline protocol pivot")
display(baseline_protocol_pivot)


Standalone baseline protocol pivot


protocol,model,primary_realistic,stress_test
0,histgb_residual_full,0.8711,0.9424
1,ridge_direct_full,0.9245,0.8463
2,structured_winner,1.0638,1.1847


In [23]:
structured_ref = (
    baseline_protocol_summary.loc[
        baseline_protocol_summary["model"] == "structured_winner",
        ["protocol", "mean_rmse"],
    ]
    .rename(columns={"mean_rmse": "structured_winner_mean_rmse"})
)

baseline_gain_table = (
    baseline_protocol_summary
    .merge(structured_ref, on="protocol", how="left")
    .assign(delta_vs_structured=lambda df: df["mean_rmse"] - df["structured_winner_mean_rmse"])
    .sort_values(["protocol", "mean_rmse"])
)

print("Standalone baseline gain/loss vs structured winner")
display(baseline_gain_table)


Standalone baseline gain/loss vs structured winner


,protocol,model,mean_rmse,min_rmse,max_rmse,structured_winner_mean_rmse,delta_vs_structured
0,primary_realistic,histgb_residual_full,0.8711,0.8171,0.8908,1.0638,-0.1927
1,primary_realistic,ridge_direct_full,0.9245,0.8397,1.0556,1.0638,-0.1393
2,primary_realistic,structured_winner,1.0638,0.7840,1.3592,1.0638,0.0000
3,stress_test,ridge_direct_full,0.8463,0.7246,1.0449,1.1847,-0.3384
4,stress_test,histgb_residual_full,0.9424,0.8574,1.1223,1.1847,-0.2423
5,stress_test,structured_winner,1.1847,1.0614,1.3163,1.1847,0.0000


In [24]:
baseline_fold_rank_view = (
    baseline_results[
        ["protocol", "fold", "model", "val_rmse"]
    ]
    .sort_values(["protocol", "fold", "val_rmse"])
    .reset_index(drop=True)
)

print("Standalone baseline fold ranking")
display(baseline_fold_rank_view)


Standalone baseline fold ranking


,protocol,fold,model,val_rmse
0,primary_realistic,1,histgb_residual_full,0.8903
1,primary_realistic,1,ridge_direct_full,1.0556
2,primary_realistic,1,structured_winner,1.3592
3,primary_realistic,2,histgb_residual_full,0.8908
4,primary_realistic,2,ridge_direct_full,0.9294
5,primary_realistic,2,structured_winner,1.0373
6,primary_realistic,3,structured_winner,0.7840
7,primary_realistic,3,ridge_direct_full,0.8733
8,primary_realistic,3,histgb_residual_full,0.8861
9,primary_realistic,4,histgb_residual_full,0.8171


In [25]:
HISTGB_ABLATION_SPECS = {
    "histgb_residual_full": {
        "excluded_groups": [],
        "notes": "Full standalone HistGB residual baseline.",
    },
    "histgb_no_structured_family": {
        "excluded_groups": ["structured_predictions", "structured_gaps"],
        "notes": "Remove all structured-model outputs and structured disagreement features.",
    },
    "histgb_no_same_date_anchor_values": {
        "excluded_groups": ["same_date_anchor_values"],
        "notes": "Remove same-date anchor IVs, distances, and same-node opposite-option IV.",
    },
    "histgb_no_support_geometry": {
        "excluded_groups": ["support_geometry"],
        "notes": "Remove local support counts, support scores, and hard-case flag.",
    },
    "histgb_no_historical_priors": {
        "excluded_groups": ["historical_priors"],
        "notes": "Remove recent/full node historical priors.",
    },
    "histgb_no_date_regime_proxies": {
        "excluded_groups": ["date_regime_proxies"],
        "notes": "Remove date-level regime / summary features.",
    },
}


In [26]:
def feature_cols_for_ablation(excluded_groups) -> list[str]:
    excluded_groups = excluded_groups or []
    excluded_cols = {
        col
        for group_name in excluded_groups
        for col in FEATURE_GROUPS[group_name]
    }
    return [col for col in FULL_HISTGB_ABLATION_FEATURES if col not in excluded_cols]


histgb_ablation_catalog = pd.DataFrame(
    [
        {
            "model": model_name,
            "excluded_groups": ", ".join(spec["excluded_groups"]) if spec["excluded_groups"] else "(none)",
            "n_raw_feature_cols": len(feature_cols_for_ablation(spec["excluded_groups"])),
            "notes": spec["notes"],
        }
        for model_name, spec in HISTGB_ABLATION_SPECS.items()
    ]
)

print("HistGB ablation catalog")
display(histgb_ablation_catalog)


HistGB ablation catalog


,model,excluded_groups,n_raw_feature_cols,notes
0,histgb_residual_full,(none),46,Full standalone HistGB residual baseline.
1,histgb_no_structured_family,"structured_predictions, structured_gaps",35,Remove all structured-model outputs and struct...
2,histgb_no_same_date_anchor_values,same_date_anchor_values,39,"Remove same-date anchor IVs, distances, and sa..."
3,histgb_no_support_geometry,support_geometry,39,"Remove local support counts, support scores, a..."
4,histgb_no_historical_priors,historical_priors,44,Remove recent/full node historical priors.
5,histgb_no_date_regime_proxies,date_regime_proxies,39,Remove date-level regime / summary features.


In [27]:
histgb_ablation_rows = []
histgb_ablation_prediction_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        train_table = table_cache[(protocol_name, fold_id)]["train"]
        val_table = table_cache[(protocol_name, fold_id)]["val"]

        for model_name, spec in HISTGB_ABLATION_SPECS.items():
            feature_cols = feature_cols_for_ablation(spec["excluded_groups"])

            summary, pred_df, model = evaluate_histgb_model(
                train_table,
                val_table,
                feature_cols=feature_cols,
                target_col="target_residual",
                learning_rate=0.05,
                max_depth=3,
                max_iter=300,
                min_samples_leaf=10,
                prediction_mode="residual",
            )

            histgb_ablation_rows.append(
                {
                    "protocol": protocol_name,
                    "fold": fold_id,
                    "model": model_name,
                    "excluded_groups": ", ".join(spec["excluded_groups"]) if spec["excluded_groups"] else "(none)",
                    "n_train_rows": len(train_table),
                    "n_val_rows": len(val_table),
                    **summary,
                }
            )

            pred_df = pred_df.copy()
            pred_df["protocol"] = protocol_name
            pred_df["fold"] = fold_id
            pred_df["model"] = model_name
            histgb_ablation_prediction_cache[(protocol_name, fold_id, model_name)] = pred_df

histgb_ablation_results = (
    pd.DataFrame(histgb_ablation_rows)
    .sort_values(["protocol", "fold", "val_rmse", "model"])
    .reset_index(drop=True)
)

print("HistGB ablation fold-level results")
display(histgb_ablation_results)


HistGB ablation fold-level results


,protocol,fold,model,excluded_groups,n_train_rows,n_val_rows,target_col,prediction_mode,learning_rate,max_depth,max_iter,min_samples_leaf,n_features_after_encoding,train_target_mean,val_rmse,structured_winner_rmse
0,primary_realistic,1,histgb_no_date_regime_proxies,date_regime_proxies,155,39,target_residual,residual,0.0500,3,300,10,58,0.0926,0.8062,1.3592
1,primary_realistic,1,histgb_no_historical_priors,historical_priors,155,39,target_residual,residual,0.0500,3,300,10,63,0.0926,0.8820,1.3592
2,primary_realistic,1,histgb_residual_full,(none),155,39,target_residual,residual,0.0500,3,300,10,65,0.0926,0.8903,1.3592
3,primary_realistic,1,histgb_no_same_date_anchor_values,same_date_anchor_values,155,39,target_residual,residual,0.0500,3,300,10,58,0.0926,0.8918,1.3592
4,primary_realistic,1,histgb_no_support_geometry,support_geometry,155,39,target_residual,residual,0.0500,3,300,10,58,0.0926,0.8995,1.3592
5,primary_realistic,1,histgb_no_structured_family,"structured_predictions, structured_gaps",155,39,target_residual,residual,0.0500,3,300,10,43,0.0926,0.9685,1.3592
6,primary_realistic,2,histgb_no_support_geometry,support_geometry,155,38,target_residual,residual,0.0500,3,300,10,58,0.1957,0.8884,1.0373
7,primary_realistic,2,histgb_no_structured_family,"structured_predictions, structured_gaps",155,38,target_residual,residual,0.0500,3,300,10,43,0.1957,0.8891,1.0373
8,primary_realistic,2,histgb_residual_full,(none),155,38,target_residual,residual,0.0500,3,300,10,65,0.1957,0.8908,1.0373
9,primary_realistic,2,histgb_no_same_date_anchor_values,same_date_anchor_values,155,38,target_residual,residual,0.0500,3,300,10,58,0.1957,0.9143,1.0373


In [28]:
histgb_ablation_protocol_summary = (
    histgb_ablation_results
    .groupby(["protocol", "model", "excluded_groups"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_rmse"])
)

print("HistGB ablation protocol summary")
display(histgb_ablation_protocol_summary)


HistGB ablation protocol summary


,protocol,model,excluded_groups,mean_rmse,min_rmse,max_rmse
2,primary_realistic,histgb_no_same_date_anchor_values,same_date_anchor_values,0.8631,0.8167,0.9143
4,primary_realistic,histgb_no_support_geometry,support_geometry,0.8685,0.7828,0.9036
0,primary_realistic,histgb_no_date_regime_proxies,date_regime_proxies,0.8705,0.8062,0.9347
5,primary_realistic,histgb_residual_full,(none),0.8711,0.8171,0.8908
1,primary_realistic,histgb_no_historical_priors,historical_priors,0.8923,0.8272,0.9326
3,primary_realistic,histgb_no_structured_family,"structured_predictions, structured_gaps",0.9369,0.8891,0.9685
6,stress_test,histgb_no_date_regime_proxies,date_regime_proxies,0.9112,0.8559,1.0014
8,stress_test,histgb_no_same_date_anchor_values,same_date_anchor_values,0.9212,0.8708,1.0586
7,stress_test,histgb_no_historical_priors,historical_priors,0.9351,0.8670,1.1142
11,stress_test,histgb_residual_full,(none),0.9424,0.8574,1.1223


In [29]:
histgb_full_ref = (
    histgb_ablation_protocol_summary.loc[
        histgb_ablation_protocol_summary["model"] == "histgb_residual_full",
        ["protocol", "mean_rmse"],
    ]
    .rename(columns={"mean_rmse": "histgb_full_mean_rmse"})
)

histgb_ablation_delta_vs_full = (
    histgb_ablation_protocol_summary
    .merge(histgb_full_ref, on="protocol", how="left")
    .assign(delta_vs_full=lambda df: df["mean_rmse"] - df["histgb_full_mean_rmse"])
    .sort_values(["protocol", "mean_rmse"])
)

print("HistGB ablation delta vs full baseline")
display(histgb_ablation_delta_vs_full)


HistGB ablation delta vs full baseline


,protocol,model,excluded_groups,mean_rmse,min_rmse,max_rmse,histgb_full_mean_rmse,delta_vs_full
0,primary_realistic,histgb_no_same_date_anchor_values,same_date_anchor_values,0.8631,0.8167,0.9143,0.8711,-0.0080
1,primary_realistic,histgb_no_support_geometry,support_geometry,0.8685,0.7828,0.9036,0.8711,-0.0025
2,primary_realistic,histgb_no_date_regime_proxies,date_regime_proxies,0.8705,0.8062,0.9347,0.8711,-0.0006
3,primary_realistic,histgb_residual_full,(none),0.8711,0.8171,0.8908,0.8711,0.0000
4,primary_realistic,histgb_no_historical_priors,historical_priors,0.8923,0.8272,0.9326,0.8711,0.0212
5,primary_realistic,histgb_no_structured_family,"structured_predictions, structured_gaps",0.9369,0.8891,0.9685,0.8711,0.0658
6,stress_test,histgb_no_date_regime_proxies,date_regime_proxies,0.9112,0.8559,1.0014,0.9424,-0.0312
7,stress_test,histgb_no_same_date_anchor_values,same_date_anchor_values,0.9212,0.8708,1.0586,0.9424,-0.0211
8,stress_test,histgb_no_historical_priors,historical_priors,0.9351,0.8670,1.1142,0.9424,-0.0073
9,stress_test,histgb_residual_full,(none),0.9424,0.8574,1.1223,0.9424,0.0000


In [30]:
histgb_ablation_importance = histgb_ablation_delta_vs_full.copy()
histgb_ablation_importance["abs_avg_drop_proxy"] = histgb_ablation_importance.groupby("model")["delta_vs_full"].transform("mean")

histgb_ablation_importance_summary = (
    histgb_ablation_importance.groupby("model", as_index=False)
    .agg(
        avg_drop_vs_full=("delta_vs_full", "mean"),
        max_drop_vs_full=("delta_vs_full", "max"),
    )
    .sort_values(["avg_drop_vs_full", "max_drop_vs_full"], ascending=False)
)

print("Feature-family importance proxy from ablations")
display(histgb_ablation_importance_summary)


Feature-family importance proxy from ablations


,model,avg_drop_vs_full,max_drop_vs_full
3,histgb_no_structured_family,0.0654,0.0658
1,histgb_no_historical_priors,0.0070,0.0212
4,histgb_no_support_geometry,0.0000,0.0026
5,histgb_residual_full,0.0000,0.0000
2,histgb_no_same_date_anchor_values,-0.0146,-0.0080
0,histgb_no_date_regime_proxies,-0.0159,-0.0006


In [31]:
def feature_cols_from_groups(
    included_groups=None,
    excluded_groups=None,
) -> list[str]:
    included_groups = included_groups or list(FEATURE_GROUPS.keys())
    excluded_groups = excluded_groups or []

    included_cols = {
        col
        for group_name in included_groups
        for col in FEATURE_GROUPS[group_name]
    }
    excluded_cols = {
        col
        for group_name in excluded_groups
        for col in FEATURE_GROUPS[group_name]
    }

    final_cols = [col for col in FULL_HISTGB_ABLATION_FEATURES if col in included_cols and col not in excluded_cols]
    return final_cols


In [32]:
HISTGB_PRUNED_CANDIDATES = {
    "histgb_residual_full": {
        "mode": "full",
        "included_groups": list(FEATURE_GROUPS.keys()),
        "excluded_groups": [],
        "notes": "Full standalone HistGB residual baseline.",
    },
    "histgb_pruned_v1_no_anchor_no_regime": {
        "mode": "exclude",
        "included_groups": list(FEATURE_GROUPS.keys()),
        "excluded_groups": ["same_date_anchor_values", "date_regime_proxies"],
        "notes": "Drop the two families that looked harmful in one-at-a-time ablations.",
    },
    "histgb_pruned_v2_no_anchor_regime_support": {
        "mode": "exclude",
        "included_groups": list(FEATURE_GROUPS.keys()),
        "excluded_groups": ["same_date_anchor_values", "date_regime_proxies", "support_geometry"],
        "notes": "Further drop support geometry, which looked nearly neutral.",
    },
    "histgb_pruned_v3_core_structured_historical": {
        "mode": "include_only",
        "included_groups": [
            "row_local_core",
            "structured_predictions",
            "structured_gaps",
            "historical_priors",
        ],
        "excluded_groups": [],
        "notes": "Lean core: row-local + structured family + historical priors only.",
    },
}


In [33]:
histgb_pruned_catalog = pd.DataFrame(
    [
        {
            "model": model_name,
            "mode": spec["mode"],
            "included_groups": ", ".join(spec["included_groups"]),
            "excluded_groups": ", ".join(spec["excluded_groups"]) if spec["excluded_groups"] else "(none)",
            "n_raw_feature_cols": len(
                feature_cols_from_groups(
                    included_groups=spec["included_groups"],
                    excluded_groups=spec["excluded_groups"],
                )
            ),
            "notes": spec["notes"],
        }
        for model_name, spec in HISTGB_PRUNED_CANDIDATES.items()
    ]
)

print("Pruned standalone HistGB candidate catalog")
display(histgb_pruned_catalog)


Pruned standalone HistGB candidate catalog


,model,mode,included_groups,excluded_groups,n_raw_feature_cols,notes
0,histgb_residual_full,full,"row_local_core, structured_predictions, struct...",(none),46,Full standalone HistGB residual baseline.
1,histgb_pruned_v1_no_anchor_no_regime,exclude,"row_local_core, structured_predictions, struct...","same_date_anchor_values, date_regime_proxies",32,Drop the two families that looked harmful in o...
2,histgb_pruned_v2_no_anchor_regime_support,exclude,"row_local_core, structured_predictions, struct...","same_date_anchor_values, date_regime_proxies, ...",25,"Further drop support geometry, which looked ne..."
3,histgb_pruned_v3_core_structured_historical,include_only,"row_local_core, structured_predictions, struct...",(none),25,Lean core: row-local + structured family + his...


In [34]:
histgb_pruned_rows = []
histgb_pruned_prediction_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        train_table = table_cache[(protocol_name, fold_id)]["train"]
        val_table = table_cache[(protocol_name, fold_id)]["val"]

        for model_name, spec in HISTGB_PRUNED_CANDIDATES.items():
            feature_cols = feature_cols_from_groups(
                included_groups=spec["included_groups"],
                excluded_groups=spec["excluded_groups"],
            )

            summary, pred_df, model = evaluate_histgb_model(
                train_table,
                val_table,
                feature_cols=feature_cols,
                target_col="target_residual",
                learning_rate=0.05,
                max_depth=3,
                max_iter=300,
                min_samples_leaf=10,
                prediction_mode="residual",
            )

            histgb_pruned_rows.append(
                {
                    "protocol": protocol_name,
                    "fold": fold_id,
                    "model": model_name,
                    "mode": spec["mode"],
                    "included_groups": ", ".join(spec["included_groups"]),
                    "excluded_groups": ", ".join(spec["excluded_groups"]) if spec["excluded_groups"] else "(none)",
                    "n_train_rows": len(train_table),
                    "n_val_rows": len(val_table),
                    **summary,
                }
            )

            pred_df = pred_df.copy()
            pred_df["protocol"] = protocol_name
            pred_df["fold"] = fold_id
            pred_df["model"] = model_name
            histgb_pruned_prediction_cache[(protocol_name, fold_id, model_name)] = pred_df

histgb_pruned_results = (
    pd.DataFrame(histgb_pruned_rows)
    .sort_values(["protocol", "fold", "val_rmse", "model"])
    .reset_index(drop=True)
)

print("Pruned standalone HistGB fold-level results")
display(histgb_pruned_results)


Pruned standalone HistGB fold-level results


,protocol,fold,model,mode,included_groups,excluded_groups,n_train_rows,n_val_rows,target_col,prediction_mode,learning_rate,max_depth,max_iter,min_samples_leaf,n_features_after_encoding,train_target_mean,val_rmse,structured_winner_rmse
0,primary_realistic,1,histgb_pruned_v1_no_anchor_no_regime,exclude,"row_local_core, structured_predictions, struct...","same_date_anchor_values, date_regime_proxies",155,39,target_residual,residual,0.0500,3,300,10,51,0.0926,0.8428,1.3592
1,primary_realistic,1,histgb_residual_full,full,"row_local_core, structured_predictions, struct...",(none),155,39,target_residual,residual,0.0500,3,300,10,65,0.0926,0.8903,1.3592
2,primary_realistic,1,histgb_pruned_v2_no_anchor_regime_support,exclude,"row_local_core, structured_predictions, struct...","same_date_anchor_values, date_regime_proxies, ...",155,39,target_residual,residual,0.0500,3,300,10,44,0.0926,0.9066,1.3592
3,primary_realistic,1,histgb_pruned_v3_core_structured_historical,include_only,"row_local_core, structured_predictions, struct...",(none),155,39,target_residual,residual,0.0500,3,300,10,44,0.0926,0.9066,1.3592
4,primary_realistic,2,histgb_residual_full,full,"row_local_core, structured_predictions, struct...",(none),155,38,target_residual,residual,0.0500,3,300,10,65,0.1957,0.8908,1.0373
5,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,exclude,"row_local_core, structured_predictions, struct...","same_date_anchor_values, date_regime_proxies",155,38,target_residual,residual,0.0500,3,300,10,51,0.1957,0.9512,1.0373
6,primary_realistic,2,histgb_pruned_v2_no_anchor_regime_support,exclude,"row_local_core, structured_predictions, struct...","same_date_anchor_values, date_regime_proxies, ...",155,38,target_residual,residual,0.0500,3,300,10,44,0.1957,1.0231,1.0373
7,primary_realistic,2,histgb_pruned_v3_core_structured_historical,include_only,"row_local_core, structured_predictions, struct...",(none),155,38,target_residual,residual,0.0500,3,300,10,44,0.1957,1.0231,1.0373
8,primary_realistic,3,histgb_pruned_v2_no_anchor_regime_support,exclude,"row_local_core, structured_predictions, struct...","same_date_anchor_values, date_regime_proxies, ...",155,39,target_residual,residual,0.0500,3,300,10,44,0.1308,0.7964,0.7840
9,primary_realistic,3,histgb_pruned_v3_core_structured_historical,include_only,"row_local_core, structured_predictions, struct...",(none),155,39,target_residual,residual,0.0500,3,300,10,44,0.1308,0.7964,0.7840


In [35]:
histgb_pruned_protocol_summary = (
    histgb_pruned_results
    .groupby(["protocol", "model", "excluded_groups"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_rmse"])
)

print("Pruned standalone HistGB protocol summary")
display(histgb_pruned_protocol_summary)


Pruned standalone HistGB protocol summary


,protocol,model,excluded_groups,mean_rmse,min_rmse,max_rmse
3,primary_realistic,histgb_residual_full,(none),0.8711,0.8171,0.8908
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,"same_date_anchor_values, date_regime_proxies",0.8725,0.8130,0.9512
1,primary_realistic,histgb_pruned_v2_no_anchor_regime_support,"same_date_anchor_values, date_regime_proxies, ...",0.9034,0.7964,1.0231
2,primary_realistic,histgb_pruned_v3_core_structured_historical,(none),0.9034,0.7964,1.0231
4,stress_test,histgb_pruned_v1_no_anchor_no_regime,"same_date_anchor_values, date_regime_proxies",0.8989,0.8522,0.9839
5,stress_test,histgb_pruned_v2_no_anchor_regime_support,"same_date_anchor_values, date_regime_proxies, ...",0.9001,0.8796,0.9318
6,stress_test,histgb_pruned_v3_core_structured_historical,(none),0.9001,0.8796,0.9318
7,stress_test,histgb_residual_full,(none),0.9424,0.8574,1.1223


In [36]:
histgb_pruned_full_ref = (
    histgb_pruned_protocol_summary.loc[
        histgb_pruned_protocol_summary["model"] == "histgb_residual_full",
        ["protocol", "mean_rmse"],
    ]
    .rename(columns={"mean_rmse": "full_histgb_mean_rmse"})
)

histgb_pruned_delta_vs_full = (
    histgb_pruned_protocol_summary
    .merge(histgb_pruned_full_ref, on="protocol", how="left")
    .assign(delta_vs_full=lambda df: df["mean_rmse"] - df["full_histgb_mean_rmse"])
    .sort_values(["protocol", "mean_rmse"])
)

print("Pruned standalone HistGB delta vs full baseline")
display(histgb_pruned_delta_vs_full)


Pruned standalone HistGB delta vs full baseline


,protocol,model,excluded_groups,mean_rmse,min_rmse,max_rmse,full_histgb_mean_rmse,delta_vs_full
0,primary_realistic,histgb_residual_full,(none),0.8711,0.8171,0.8908,0.8711,0.0000
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,"same_date_anchor_values, date_regime_proxies",0.8725,0.8130,0.9512,0.8711,0.0015
2,primary_realistic,histgb_pruned_v2_no_anchor_regime_support,"same_date_anchor_values, date_regime_proxies, ...",0.9034,0.7964,1.0231,0.8711,0.0324
3,primary_realistic,histgb_pruned_v3_core_structured_historical,(none),0.9034,0.7964,1.0231,0.8711,0.0324
4,stress_test,histgb_pruned_v1_no_anchor_no_regime,"same_date_anchor_values, date_regime_proxies",0.8989,0.8522,0.9839,0.9424,-0.0435
5,stress_test,histgb_pruned_v2_no_anchor_regime_support,"same_date_anchor_values, date_regime_proxies, ...",0.9001,0.8796,0.9318,0.9424,-0.0423
6,stress_test,histgb_pruned_v3_core_structured_historical,(none),0.9001,0.8796,0.9318,0.9424,-0.0423
7,stress_test,histgb_residual_full,(none),0.9424,0.8574,1.1223,0.9424,0.0000


In [37]:
histgb_pruned_pivot = (
    histgb_pruned_protocol_summary
    .pivot(index="model", columns="protocol", values="mean_rmse")
    .reset_index()
)

histgb_pruned_delta_pivot = (
    histgb_pruned_delta_vs_full[["protocol", "model", "delta_vs_full"]]
    .pivot(index="model", columns="protocol", values="delta_vs_full")
    .reset_index()
    .rename(
        columns={
            "primary_realistic": "delta_vs_full_primary_realistic",
            "stress_test": "delta_vs_full_stress_test",
        }
    )
)

histgb_pruned_decision = (
    histgb_pruned_pivot
    .merge(histgb_pruned_delta_pivot, on="model", how="left")
)

histgb_pruned_decision["avg_two_protocols"] = (
    histgb_pruned_decision["primary_realistic"] + histgb_pruned_decision["stress_test"]
) / 2.0

histgb_pruned_decision["max_two_protocols"] = histgb_pruned_decision[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("Pruned standalone HistGB decision table")
display(histgb_pruned_decision.sort_values(["max_two_protocols", "avg_two_protocols"]))


Pruned standalone HistGB decision table


protocol,model,primary_realistic,stress_test,delta_vs_full_primary_realistic,delta_vs_full_stress_test,avg_two_protocols,max_two_protocols
0,histgb_pruned_v1_no_anchor_no_regime,0.8725,0.8989,0.0015,-0.0435,0.8857,0.8989
1,histgb_pruned_v2_no_anchor_regime_support,0.9034,0.9001,0.0324,-0.0423,0.9018,0.9034
2,histgb_pruned_v3_core_structured_historical,0.9034,0.9001,0.0324,-0.0423,0.9018,0.9034
3,histgb_residual_full,0.8711,0.9424,0.0000,0.0000,0.9067,0.9424


In [38]:
pruned_winner_names = histgb_pruned_decision.sort_values(["max_two_protocols", "avg_two_protocols"])["model"].tolist()
best_pruned_name = pruned_winner_names[0]

best_pruned_protocol_summary = histgb_pruned_protocol_summary.loc[
    histgb_pruned_protocol_summary["model"] == best_pruned_name,
    ["protocol", "mean_rmse"]
].rename(columns={"mean_rmse": "best_pruned_histgb_mean_rmse"})

reference_compare = (
    baseline_protocol_summary[["protocol", "model", "mean_rmse"]]
    .merge(best_pruned_protocol_summary, on="protocol", how="left")
)

print("Reference comparison vs best pruned HistGB candidate")
display(reference_compare.sort_values(["protocol", "mean_rmse"]))
print("Best pruned candidate:", best_pruned_name)


Reference comparison vs best pruned HistGB candidate


,protocol,model,mean_rmse,best_pruned_histgb_mean_rmse
0,primary_realistic,histgb_residual_full,0.8711,0.8725
1,primary_realistic,ridge_direct_full,0.9245,0.8725
2,primary_realistic,structured_winner,1.0638,0.8725
3,stress_test,ridge_direct_full,0.8463,0.8989
4,stress_test,histgb_residual_full,0.9424,0.8989
5,stress_test,structured_winner,1.1847,0.8989


Best pruned candidate: histgb_pruned_v1_no_anchor_no_regime


In [39]:
HISTGB_COMPARE_MODELS = [
    "histgb_residual_full",
    "histgb_pruned_v1_no_anchor_no_regime",
]

histgb_compare_rows = []

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])
        val_table = table_cache[(protocol_name, fold_id)]["val"].copy()

        base = val_table[
            [
                "row_id",
                "target_date",
                "target_iv",
                "maturity_label",
                "maturity_days",
                "option_type",
                "moneyness",
                "wing_center_bucket",
                "hard_case",
                "structured_winner_source",
            ]
        ].copy()

        base["hard_case_bucket"] = np.where(base["hard_case"], "hard_case", "non_hard_case")

        for model_name in HISTGB_COMPARE_MODELS:
            pred_df = histgb_pruned_prediction_cache[(protocol_name, fold_id, model_name)].copy()

            merged = base.merge(
                pred_df[["row_id", "iv_pred"]],
                on="row_id",
                how="left",
            )
            merged["protocol"] = protocol_name
            merged["fold"] = fold_id
            merged["model"] = model_name
            merged["abs_error"] = (merged["target_iv"] - merged["iv_pred"]).abs()
            merged["sq_error"] = (merged["target_iv"] - merged["iv_pred"]) ** 2

            histgb_compare_rows.append(merged)

histgb_compare_df = pd.concat(histgb_compare_rows, ignore_index=True)

print("Standalone HistGB comparison dataframe")
display(histgb_compare_df.head())


Standalone HistGB comparison dataframe


,row_id,target_date,target_iv,maturity_label,maturity_days,option_type,moneyness,wing_center_bucket,hard_case,structured_winner_source,hard_case_bucket,iv_pred,protocol,fold,model,abs_error,sq_error
0,9262,2025-04-21,19.2304,1M,30,call,1.1000,center,False,structured_region_blend_callput_shrink,non_hard_case,19.2284,primary_realistic,1,histgb_residual_full,0.0020,0.0000
1,9245,2025-04-21,24.2840,1M,30,put,0.8750,wing,False,quadratic_smile_only,non_hard_case,25.7073,primary_realistic,1,histgb_residual_full,1.4233,2.0258
2,9294,2025-04-21,19.0918,2M,60,call,1.1250,wing,False,structured_region_blend_callput_shrink,non_hard_case,18.2645,primary_realistic,1,histgb_residual_full,0.8273,0.6845
3,9281,2025-04-21,20.2648,2M,60,put,0.9500,center,False,structured_region_blend_callput_shrink,non_hard_case,21.0538,primary_realistic,1,histgb_residual_full,0.7890,0.6225
4,9312,2025-04-21,18.7062,3M,91,call,0.9750,center,False,structured_region_blend_callput_shrink,non_hard_case,19.3050,primary_realistic,1,histgb_residual_full,0.5988,0.3586


In [40]:
def slice_error_table_local(df: pd.DataFrame, group_cols) -> pd.DataFrame:
    group_cols = list(group_cols)
    out = (
        df.groupby(["protocol", "model", *group_cols], dropna=False)
        .agg(
            n=("row_id", "size"),
            mae=("abs_error", "mean"),
            mean_sq_error=("sq_error", "mean"),
        )
        .reset_index()
    )
    out["rmse"] = np.sqrt(out["mean_sq_error"])
    return out.drop(columns=["mean_sq_error"]).sort_values(["protocol", "model", *group_cols]).reset_index(drop=True)


In [41]:
histgb_wing_center = slice_error_table_local(
    histgb_compare_df,
    group_cols=["wing_center_bucket"],
)

print("Full vs pruned HistGB | wing vs center")
display(histgb_wing_center)


Full vs pruned HistGB | wing vs center


,protocol,model,wing_center_bucket,n,mae,rmse
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,center,65,0.6043,0.7645
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,wing,91,0.7871,0.9439
2,primary_realistic,histgb_residual_full,center,65,0.5462,0.6949
3,primary_realistic,histgb_residual_full,wing,91,0.8124,0.9778
4,stress_test,histgb_pruned_v1_no_anchor_no_regime,center,49,0.5953,0.7387
5,stress_test,histgb_pruned_v1_no_anchor_no_regime,wing,107,0.7643,0.9651
6,stress_test,histgb_residual_full,center,49,0.7077,0.8245
7,stress_test,histgb_residual_full,wing,107,0.7882,0.9999


In [42]:
histgb_maturity = slice_error_table_local(
    histgb_compare_df,
    group_cols=["maturity_label"],
)

print("Full vs pruned HistGB | maturity")
display(histgb_maturity)


Full vs pruned HistGB | maturity


,protocol,model,maturity_label,n,mae,rmse
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,1M,40,0.7523,0.9516
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,2M,39,0.7090,0.8688
2,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,3M,39,0.7484,0.8758
3,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,6M,38,0.6312,0.7862
4,primary_realistic,histgb_residual_full,1M,40,0.7552,0.9594
5,primary_realistic,histgb_residual_full,2M,39,0.7200,0.9207
6,primary_realistic,histgb_residual_full,3M,39,0.6800,0.7890
7,primary_realistic,histgb_residual_full,6M,38,0.6480,0.7987
8,stress_test,histgb_pruned_v1_no_anchor_no_regime,1M,40,0.6063,0.7883
9,stress_test,histgb_pruned_v1_no_anchor_no_regime,2M,39,0.8046,1.0110


In [43]:
histgb_maturity = slice_error_table_local(
    histgb_compare_df,
    group_cols=["maturity_label"],
)

print("Full vs pruned HistGB | maturity")
display(histgb_maturity)


Full vs pruned HistGB | maturity


,protocol,model,maturity_label,n,mae,rmse
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,1M,40,0.7523,0.9516
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,2M,39,0.7090,0.8688
2,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,3M,39,0.7484,0.8758
3,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,6M,38,0.6312,0.7862
4,primary_realistic,histgb_residual_full,1M,40,0.7552,0.9594
5,primary_realistic,histgb_residual_full,2M,39,0.7200,0.9207
6,primary_realistic,histgb_residual_full,3M,39,0.6800,0.7890
7,primary_realistic,histgb_residual_full,6M,38,0.6480,0.7987
8,stress_test,histgb_pruned_v1_no_anchor_no_regime,1M,40,0.6063,0.7883
9,stress_test,histgb_pruned_v1_no_anchor_no_regime,2M,39,0.8046,1.0110


In [45]:
histgb_source = slice_error_table_local(
    histgb_compare_df,
    group_cols=["structured_winner_source"],
)

print("Full vs pruned HistGB | source")
display(histgb_source)


Full vs pruned HistGB | source


,protocol,model,structured_winner_source,n,mae,rmse
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,quadratic_smile_only,34,0.7422,0.8992
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,same_date_linear_interp,18,0.8202,1.0868
2,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,structured_region_blend_callput_shrink,89,0.6849,0.8257
3,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,structured_region_blend_center,4,0.6984,0.8361
4,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,structured_region_blend_wing,1,0.5387,0.5387
5,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,tv_maturity_only,10,0.6622,0.8087
6,primary_realistic,histgb_residual_full,quadratic_smile_only,34,0.7767,0.9232
7,primary_realistic,histgb_residual_full,same_date_linear_interp,18,0.8294,1.1380
8,primary_realistic,histgb_residual_full,structured_region_blend_callput_shrink,89,0.6600,0.8000
9,primary_realistic,histgb_residual_full,structured_region_blend_center,4,0.4742,0.6751


In [46]:
histgb_source_min10 = histgb_source.loc[histgb_source["n"] >= 10].copy()

print("Full vs pruned HistGB | source | n >= 10")
display(histgb_source_min10)


Full vs pruned HistGB | source | n >= 10


,protocol,model,structured_winner_source,n,mae,rmse
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,quadratic_smile_only,34,0.7422,0.8992
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,same_date_linear_interp,18,0.8202,1.0868
2,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,structured_region_blend_callput_shrink,89,0.6849,0.8257
5,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,tv_maturity_only,10,0.6622,0.8087
6,primary_realistic,histgb_residual_full,quadratic_smile_only,34,0.7767,0.9232
7,primary_realistic,histgb_residual_full,same_date_linear_interp,18,0.8294,1.1380
8,primary_realistic,histgb_residual_full,structured_region_blend_callput_shrink,89,0.6600,0.8000
11,primary_realistic,histgb_residual_full,tv_maturity_only,10,0.7056,0.8464
12,stress_test,histgb_pruned_v1_no_anchor_no_regime,quadratic_smile_only,41,0.6193,0.8390
13,stress_test,histgb_pruned_v1_no_anchor_no_regime,same_date_linear_interp,30,0.6451,0.8564


In [47]:
def head_to_head_delta(slice_table: pd.DataFrame, slice_cols) -> pd.DataFrame:
    slice_cols = list(slice_cols)

    pivot = (
        slice_table.pivot_table(
            index=["protocol", *slice_cols],
            columns="model",
            values="rmse",
        )
        .reset_index()
    )

    pivot["pruned_minus_full"] = (
        pivot["histgb_pruned_v1_no_anchor_no_regime"]
        - pivot["histgb_residual_full"]
    )
    return pivot.sort_values(["protocol", *slice_cols]).reset_index(drop=True)


In [48]:
histgb_wing_center_delta = head_to_head_delta(histgb_wing_center, ["wing_center_bucket"])
histgb_maturity_delta = head_to_head_delta(histgb_maturity, ["maturity_label"])
histgb_source_delta = head_to_head_delta(histgb_source_min10, ["structured_winner_source"])

print("Full vs pruned HistGB | wing/center delta")
display(histgb_wing_center_delta)

print("Full vs pruned HistGB | maturity delta")
display(histgb_maturity_delta)

print("Full vs pruned HistGB | source delta | n >= 10")
display(histgb_source_delta)


Full vs pruned HistGB | wing/center delta


model,protocol,wing_center_bucket,histgb_pruned_v1_no_anchor_no_regime,histgb_residual_full,pruned_minus_full
0,primary_realistic,center,0.7645,0.6949,0.0696
1,primary_realistic,wing,0.9439,0.9778,-0.0339
2,stress_test,center,0.7387,0.8245,-0.0858
3,stress_test,wing,0.9651,0.9999,-0.0348


Full vs pruned HistGB | maturity delta


model,protocol,maturity_label,histgb_pruned_v1_no_anchor_no_regime,histgb_residual_full,pruned_minus_full
0,primary_realistic,1M,0.9516,0.9594,-0.0078
1,primary_realistic,2M,0.8688,0.9207,-0.0519
2,primary_realistic,3M,0.8758,0.7890,0.0868
3,primary_realistic,6M,0.7862,0.7987,-0.0126
4,stress_test,1M,0.7883,0.8152,-0.0269
5,stress_test,2M,1.0110,1.0792,-0.0682
6,stress_test,3M,0.7919,0.8167,-0.0249
7,stress_test,6M,0.9898,1.0547,-0.0649


Full vs pruned HistGB | source delta | n >= 10


model,protocol,structured_winner_source,histgb_pruned_v1_no_anchor_no_regime,histgb_residual_full,pruned_minus_full
0,primary_realistic,quadratic_smile_only,0.8992,0.9232,-0.0240
1,primary_realistic,same_date_linear_interp,1.0868,1.1380,-0.0511
2,primary_realistic,structured_region_blend_callput_shrink,0.8257,0.8000,0.0257
3,primary_realistic,tv_maturity_only,0.8087,0.8464,-0.0376
4,stress_test,quadratic_smile_only,0.8390,0.8910,-0.0520
5,stress_test,same_date_linear_interp,0.8564,0.9612,-0.1047
6,stress_test,structured_region_blend_callput_shrink,0.9102,0.9210,-0.0107
7,stress_test,tv_maturity_only,1.1195,1.2343,-0.1148


In [49]:
histgb_compare_summary = pd.DataFrame(
    [
        {
            "model": "histgb_residual_full",
            "primary_realistic": 0.8711,
            "stress_test": 0.9424,
        },
        {
            "model": "histgb_pruned_v1_no_anchor_no_regime",
            "primary_realistic": 0.8725,
            "stress_test": 0.8989,
        },
    ]
)

histgb_compare_summary["avg_two_protocols"] = (
    histgb_compare_summary["primary_realistic"] + histgb_compare_summary["stress_test"]
) / 2.0

histgb_compare_summary["max_two_protocols"] = histgb_compare_summary[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("Standalone full vs pruned HistGB summary")
display(histgb_compare_summary)


Standalone full vs pruned HistGB summary


,model,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
0,histgb_residual_full,0.8711,0.9424,0.9067,0.9424
1,histgb_pruned_v1_no_anchor_no_regime,0.8725,0.8989,0.8857,0.8989


In [50]:
standalone_carry_forward = pd.DataFrame(
    [
        {
            "model": "structured_winner",
            "role": "benchmark",
            "family": "structured",
            "target_mode": "direct_iv",
            "notes": "Locked Phase 4 benchmark.",
        },
        {
            "model": "ridge_direct_full",
            "role": "best_linear_reference",
            "family": "linear_ml",
            "target_mode": "direct_iv",
            "notes": "Best conservative standalone linear reference.",
        },
        {
            "model": "histgb_residual_full",
            "role": "full_nonlinear_reference",
            "family": "tree_ml",
            "target_mode": "residual",
            "notes": "Original standalone nonlinear residual reference from 05_0.",
        },
        {
            "model": "histgb_pruned_v1_no_anchor_no_regime",
            "role": "best_pruned_nonlinear_reference",
            "family": "tree_ml",
            "target_mode": "residual",
            "notes": "Best standalone nonlinear reference after feature-family pruning.",
        },
    ]
)

standalone_perf_base = baseline_protocol_summary[
    ["protocol", "model", "mean_rmse"]
].copy()

pruned_perf_base = histgb_pruned_protocol_summary[
    ["protocol", "model", "mean_rmse"]
].copy()

standalone_perf_all = pd.concat(
    [standalone_perf_base, pruned_perf_base],
    ignore_index=True,
).drop_duplicates(subset=["protocol", "model"], keep="last")

standalone_perf_pivot = (
    standalone_perf_all
    .pivot(index="model", columns="protocol", values="mean_rmse")
    .reset_index()
)

standalone_perf_pivot["avg_two_protocols"] = (
    standalone_perf_pivot["primary_realistic"] + standalone_perf_pivot["stress_test"]
) / 2.0

standalone_perf_pivot["max_two_protocols"] = standalone_perf_pivot[
    ["primary_realistic", "stress_test"]
].max(axis=1)

standalone_final_registry = (
    standalone_carry_forward
    .merge(standalone_perf_pivot, on="model", how="left")
    .sort_values(["max_two_protocols", "avg_two_protocols"])
    .reset_index(drop=True)
)

print("05_1 standalone carry-forward registry")
display(standalone_final_registry)


05_1 standalone carry-forward registry


,model,role,family,target_mode,notes,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
0,histgb_pruned_v1_no_anchor_no_regime,best_pruned_nonlinear_reference,tree_ml,residual,Best standalone nonlinear reference after feat...,0.8725,0.8989,0.8857,0.8989
1,ridge_direct_full,best_linear_reference,linear_ml,direct_iv,Best conservative standalone linear reference.,0.9245,0.8463,0.8854,0.9245
2,histgb_residual_full,full_nonlinear_reference,tree_ml,residual,Original standalone nonlinear residual referen...,0.8711,0.9424,0.9067,0.9424
3,structured_winner,benchmark,structured,direct_iv,Locked Phase 4 benchmark.,1.0638,1.1847,1.1243,1.1847


In [51]:
structured_ref = standalone_perf_all.loc[
    standalone_perf_all["model"] == "structured_winner",
    ["protocol", "mean_rmse"],
].rename(columns={"mean_rmse": "structured_mean_rmse"})

full_histgb_ref = standalone_perf_all.loc[
    standalone_perf_all["model"] == "histgb_residual_full",
    ["protocol", "mean_rmse"],
].rename(columns={"mean_rmse": "full_histgb_mean_rmse"})

standalone_delta_table = (
    standalone_perf_all
    .merge(structured_ref, on="protocol", how="left")
    .merge(full_histgb_ref, on="protocol", how="left")
    .assign(
        delta_vs_structured=lambda df: df["mean_rmse"] - df["structured_mean_rmse"],
        delta_vs_full_histgb=lambda df: df["mean_rmse"] - df["full_histgb_mean_rmse"],
    )
    .sort_values(["protocol", "mean_rmse"])
)

print("05_1 standalone delta table")
display(standalone_delta_table)


05_1 standalone delta table


,protocol,model,mean_rmse,structured_mean_rmse,full_histgb_mean_rmse,delta_vs_structured,delta_vs_full_histgb
4,primary_realistic,histgb_residual_full,0.8711,1.0638,0.8711,-0.1927,0.0000
5,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,0.8725,1.0638,0.8711,-0.1913,0.0015
6,primary_realistic,histgb_pruned_v2_no_anchor_regime_support,0.9034,1.0638,0.8711,-0.1604,0.0324
7,primary_realistic,histgb_pruned_v3_core_structured_historical,0.9034,1.0638,0.8711,-0.1604,0.0324
0,primary_realistic,ridge_direct_full,0.9245,1.0638,0.8711,-0.1393,0.0534
1,primary_realistic,structured_winner,1.0638,1.0638,0.8711,0.0000,0.1927
2,stress_test,ridge_direct_full,0.8463,1.1847,0.9424,-0.3384,-0.0960
8,stress_test,histgb_pruned_v1_no_anchor_no_regime,0.8989,1.1847,0.9424,-0.2858,-0.0435
9,stress_test,histgb_pruned_v2_no_anchor_regime_support,0.9001,1.1847,0.9424,-0.2846,-0.0423
10,stress_test,histgb_pruned_v3_core_structured_historical,0.9001,1.1847,0.9424,-0.2846,-0.0423


## Phase 5 follow-up: feature-family ablations on the standalone HistGB residual model

This notebook is a follow-up to `05_0_ml_feature_pipeline.ipynb`.

Its purpose was to complete the missing explanatory part of Phase 5:

- test which feature families actually drive the standalone nonlinear ML gains
- determine whether the standalone `histgb_residual_full` reference should be simplified
- keep the hybrid leader from `05_0` outside the scope of this notebook

---

## 1. What this notebook tested

The primary standalone model studied here was:

- `histgb_residual_full`

This notebook rebuilt the same leakage-safe train / validation tables as `05_0`, then ran:

1. standalone baseline reproduction
2. one-at-a-time feature-family ablations
3. cumulative pruned standalone HistGB candidates

The feature-family groups tested were:

- row-local core
- structured predictions
- structured gaps
- same-date anchor-value features
- support geometry
- historical priors
- date-level regime proxies

---

## 2. Main ablation result

The most important one-at-a-time ablation result is:

### The structured family is the dominant driver of standalone HistGB performance

Removing:
- `structured_predictions`
- `structured_gaps`

caused the largest degradation on both protocols.

This shows that the standalone HistGB residual model is still fundamentally a:

**structured-model corrector**

rather than a model that independently reconstructs the IV surface from raw local features.

---

## 3. Secondary findings

### Historical priors
Historical priors were somewhat useful, but their benefit was smaller and less consistent than the structured family.

### Support geometry
Support-geometry features had very small incremental effect in this fixed standalone HistGB setup.

### Same-date anchor-value features
Removing the raw same-date anchor-value features improved performance, especially under the conservative protocol.

### Date-level regime proxies
Removing the regime/date-summary proxy features also improved performance, especially under the conservative protocol.

This does not mean these features are universally useless.
It means that in this specific standalone HistGB residual setup, they were not adding enough incremental signal beyond the stronger structured layer and were acting mostly as redundant or noisy inputs.

---

## 4. Best pruned standalone nonlinear reference

The best pruned standalone nonlinear model from this notebook is:

- `histgb_pruned_v1_no_anchor_no_regime`

Rule:
- same standalone HistGB residual setup
- remove:
  - `same_date_anchor_values`
  - `date_regime_proxies`

Performance:

- full `histgb_residual_full`
  - `primary_realistic = 0.8711`
  - `stress_test = 0.9424`

- `histgb_pruned_v1_no_anchor_no_regime`
  - `primary_realistic = 0.8725`
  - `stress_test = 0.8989`

Interpretation:
- essentially tied on `primary_realistic`
- materially better on `stress_test`

So this pruned model becomes the cleaner standalone nonlinear reference going forward.

---

## 5. Where the pruning helps

The pruned model helps mainly by improving robustness in harder settings.

### Wing vs center
- under `stress_test`, the pruned model improves both center and wing slices
- under `primary_realistic`, it gives up some center performance but improves the wings

### Maturity
- under `stress_test`, the pruned model improves all maturity buckets
- under `primary_realistic`, the only meaningful giveback is mostly in `3M`

### Structured source buckets
The pruned model improves several important source buckets, especially:
- `same_date_linear_interp`
- `tv_maturity_only`
- `quadratic_smile_only`

This again suggests the removed families were adding noise more than useful incremental signal in the harder settings.

---

## 6. Final conclusion from this notebook

This notebook completes the missing explanatory part of Phase 5.

The main conclusions are:

1. The standalone nonlinear ML gains are driven primarily by the **structured feature family**.
2. The standalone HistGB residual model works best as a **structured-model corrector**.
3. Raw same-date anchor-value features and date-level regime proxies are not incrementally helping this standalone model.
4. The best standalone nonlinear reference is now the simplified:
   - `histgb_pruned_v1_no_anchor_no_regime`

So `05_1` does not change the broader conclusion from `05_0` that selective hybrid correction is promising.

What it does do is make the standalone story cleaner:

- benchmark:
  - `structured_winner`
- best linear standalone reference:
  - `ridge_direct_full`
- best standalone nonlinear residual reference:
  - `histgb_pruned_v1_no_anchor_no_regime`

That is the main handoff from this notebook.
